# AIQUEST LTX-2.5 22B Distilled — Backend-Automated Pipeline

Rebuilt on top of the more complete upstream notebook (dynamic VRAM budget
sizing, native I2V black-frame fix, resolution snapping — no more Step 3.5/3.6
string patches for those). Automation layer (dataset-based job ingestion,
scene chaining, curated output, auto-shutdown) is fully rewritten in Step 4.

**Run order:** Step 1 -> Model weights (cached dataset) -> Step 2 -> Step 3 ->
Step 3.5 hotfix -> Headless launch patch -> Step 4 (headless batch).

Attach both the `deepbeepmeep` model-weights dataset and make sure this
kernel's Kaggle API credentials can see the backend's `ltx-job-request`
dataset before running Step 4.


In [ ]:
#@title Step 1: Environment Setup, Dependencies & Dual GPU Optimization (GPU T4 x2)
import os
import sys
import gc
import psutil
import torch
import subprocess

print('=== Kaggle Dual GPU Environment Setup (GPU T4 x2) ===')
ram = psutil.virtual_memory()
print(f'System RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available')

gpu_count = torch.cuda.device_count()
print(f'Detected GPUs: {gpu_count}')
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    free_vram = torch.cuda.mem_get_info(i)[0] / 1024**3
    total_vram = props.total_memory / 1024**3
    print(f'  - cuda:{i} ({props.name}): {free_vram:.1f} GB free / {total_vram:.1f} GB total (Capability {props.major}.{props.minor})')

if gpu_count < 2:
    print('\n⚠️ WARNING: Less than 2 GPUs detected! Please verify Settings → Accelerator → GPU T4 x2')
else:
    print('\n✅ Dual GPU Configuration (GPU T4 x2) Verified!')

# System cache clearing and overcommit settings
os.system('rm -rf /kaggle/tmp/* /tmp/*')
os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

# PyTorch allocator & tokenizer memory optimizations
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TORCH_CUDNN_V8_API_ENABLED'] = '1'

# Clone or clean Wan2GP framework with LTX-2.5 support
#
# Pinned to a known-good commit instead of floating on `main`. Wan2GP is
# actively developed, and an untested upstream commit can break model loading
# outright with zero warning - on 2026-10-07 a commit landed that made
# ltx2_handler.py require sys.modules["wgp"] to already exist, which is only
# true when Wan2GP's own wgp.py is the running entry point, not when used
# standalone the way this notebook does. Every prior run before that date
# worked fine on the exact same code; nothing in this notebook broke - an
# unrelated upstream commit did. Bump this pin deliberately (and test the
# bump in isolation before relying on it) rather than tracking main blindly.
WAN2GP_PINNED_COMMIT = "f02a38275fa43057ea75814e8ef4368e5530dcf2"  # 2026-10-06, last known-good

if not os.path.exists('Wan2GP'):
    print('Cloning Wan2GP repository...')
    subprocess.run(['git', 'clone', 'https://github.com/DeepBeepMeep/Wan2GP.git'], check=True)
    print('✓ Wan2GP cloned successfully!')
else:
    print('✓ Wan2GP repository exists. Resetting to clean git state...')
    subprocess.run(['git', 'checkout', '.'], cwd='Wan2GP', check=False)
    subprocess.run(['git', 'fetch', 'origin'], cwd='Wan2GP', check=False)

print(f'Pinning Wan2GP to known-good commit {WAN2GP_PINNED_COMMIT}...')
subprocess.run(['git', 'checkout', WAN2GP_PINNED_COMMIT], cwd='Wan2GP', check=True)
print('✓ Wan2GP pinned to a known-good commit.')

# Install complete Wan2GP dependencies
print('Installing all core Wan2GP dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'sageattention'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--timeout', '180',
    '-r', 'Wan2GP/requirements.txt'
], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai-whisper',
    'ffmpeg-python',
    'onnxruntime',
    'rembg',
    'gradio>=4.40.0',
    'bitsandbytes',
    'soundfile',
    'opencv-python'
], check=False)

# Apply automatic high-speed and visual quality patches to Wan2GP
print('Applying high-speed FP16 & crystal-clear quality patches to Wan2GP...')
for base_dir in ['Wan2GP', '.']:
    # 1. High-Speed FP16 on T4 Tensor Cores (~24s/step)
    distilled_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_pipelines', 'distilled.py')
    if os.path.isfile(distilled_file):
        with open(distilled_file, 'r', encoding='utf-8') as f:
            dist_code = f.read()
        dist_code = dist_code.replace("self.dtype = torch.bfloat16", "self.dtype = getattr(models, 'dtype', torch.float16)")
        dist_code = dist_code.replace("dtype = torch.bfloat16", "dtype = getattr(self, 'dtype', torch.float16)")
        with open(distilled_file, 'w', encoding='utf-8') as f:
            f.write(dist_code)

    # 2. PixelNorm Float32 precision (fixes FP16 overflow to solid gray background)
    norm_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'common', 'normalization.py')
    if os.path.isfile(norm_file):
        with open(norm_file, 'r', encoding='utf-8') as f:
            norm_code = f.read()
        if 'x_float = x.float()' not in norm_code:
            norm_code = norm_code.replace(
                "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\n        # Normalize by the root-mean-square (RMS).\n        rms = torch.sqrt(mean_sq + self.eps)\n        return x / rms",
                "        orig_dtype = x.dtype\n        x_float = x.float()\n        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\n        rms = torch.sqrt(mean_sq + self.eps)\n        return (x_float / rms).to(orig_dtype)"
            )
            with open(norm_file, 'w', encoding='utf-8') as f:
                f.write(norm_code)

    # 3. Gemma Feature Extractor RMS float32 precision
    fe_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'text_encoders', 'gemma', 'feature_extractor.py')
    if os.path.isfile(fe_file):
        with open(fe_file, 'r', encoding='utf-8') as f:
            fe_code = f.read()
        if 'enc_f = encoded_text.float()' not in fe_code:
            fe_code = fe_code.replace(
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    b, t, d, l = encoded_text.shape\n    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\n    normed = encoded_text * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    orig_dtype = encoded_text.dtype\n    enc_f = encoded_text.float()\n    b, t, d, l = enc_f.shape\n    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\n    normed = enc_f * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l).to(orig_dtype)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))"
            )
            with open(fe_file, 'w', encoding='utf-8') as f:
                f.write(fe_code)

    # 4. VAE CausalConv3d auto-cast
    conv_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'video_vae', 'convolution.py')
    if os.path.isfile(conv_file):
        with open(conv_file, 'r', encoding='utf-8') as f:
            conv_code = f.read()
        if 'x = self.conv(x)' in conv_code and 'x.dtype != self.conv.weight.dtype' not in conv_code:
            conv_code = conv_code.replace(
                "        x = self.conv(x)",
                "        if hasattr(self, 'conv') and hasattr(self.conv, 'weight') and x.dtype != self.conv.weight.dtype:\n            x = x.to(dtype=self.conv.weight.dtype)\n        x = self.conv(x)"
            )
            with open(conv_file, 'w', encoding='utf-8') as f:
                f.write(conv_code)

    # 5. Audio VAE & Vocoder buffer auto-casts
    audio_vae_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'audio_vae', 'audio_vae.py')
    if os.path.isfile(audio_vae_file):
        with open(audio_vae_file, 'r', encoding='utf-8') as f:
            audio_vae_code = f.read()
        if 'decoded_audio = audio_decoder(latent)' in audio_vae_code and 'latent.to(dtype=' not in audio_vae_code:
            audio_vae_code = audio_vae_code.replace(
                "    decoded_audio = audio_decoder(latent)",
                "    if hasattr(audio_decoder, 'conv_in') and hasattr(audio_decoder.conv_in, 'weight'):\n        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\n    decoded_audio = audio_decoder(latent)\n    if hasattr(vocoder, 'vocoder') and hasattr(vocoder.vocoder, 'conv_pre'):\n        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)"
            )
            with open(audio_vae_file, 'w', encoding='utf-8') as f:
                f.write(audio_vae_code)

    vocoder_file = os.path.join(base_dir, 'models', 'ltx2', 'ltx_core', 'model', 'audio_vae', 'vocoder.py')
    if os.path.isfile(vocoder_file):
        with open(vocoder_file, 'r', encoding='utf-8') as f:
            vocoder_code = f.read()
        vocoder_code = vocoder_code.replace(
            "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)",
            "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\n        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)"
        )
        vocoder_code = vocoder_code.replace(
            "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
            "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\n        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)"
        )
        with open(vocoder_file, 'w', encoding='utf-8') as f:
            f.write(vocoder_code)

print('✅ Environment, Dependencies & Quality Patches Complete!')

In [ ]:
# === Run this in Kaggle AFTER attaching the deepbeepmeep dataset, BEFORE
# the original script's "Step 2: Download..." cell. It recreates exactly
# the file layout Step 2 already checks for with "if os.path.exists(...):
# skip" — so Step 2 will find everything already there and do no downloading.
#
# Still run the ORIGINAL Step 1 (env setup + Wan2GP clone) unmodified first.

import os

# Adjust after checking /kaggle/input:
DATASET_INPUT = "/kaggle/input/datasets/nsisongokon/ltx-2-5-deepbeepmeep-int8"

MODEL_DIR = os.path.abspath("Wan2GP/models")
CKPTS_DIR = os.path.abspath("Wan2GP/ckpts")
TMP_DIR = "/tmp/models"   # matches where Step 2 expects the transformer specifically
GEMMA4_SUBFOLDER = "gemma4-12b-ltx-v1"

for d in (MODEL_DIR, CKPTS_DIR, TMP_DIR,
          os.path.join(MODEL_DIR, GEMMA4_SUBFOLDER),
          os.path.join(CKPTS_DIR, GEMMA4_SUBFOLDER)):
    os.makedirs(d, exist_ok=True)

TRANSFORMER_FILE = "ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"

linked = 0
for fname in os.listdir(DATASET_INPUT):
    src = os.path.join(DATASET_INPUT, fname)
    if not os.path.isfile(src):
        continue

    if fname == TRANSFORMER_FILE:
        # Step 2 expects this one specifically staged in /tmp/models, then
        # symlinked into models/ and ckpts/.
        targets = [
            os.path.join(TMP_DIR, fname),
            os.path.join(MODEL_DIR, fname),
            os.path.join(CKPTS_DIR, fname),
        ]
    elif fname.startswith(f"{GEMMA4_SUBFOLDER}__"):
        # Un-flatten: strip the prefix, put it back in the gemma subfolder.
        real_name = fname[len(GEMMA4_SUBFOLDER) + 2:]
        targets = [
            os.path.join(MODEL_DIR, GEMMA4_SUBFOLDER, real_name),
            os.path.join(CKPTS_DIR, GEMMA4_SUBFOLDER, real_name),
        ]
    elif fname in ("dataset-metadata.json",):
        continue
    else:
        # Ordinary companion file — flat in both models/ and ckpts/.
        targets = [
            os.path.join(MODEL_DIR, fname),
            os.path.join(CKPTS_DIR, fname),
        ]

    for dst in targets:
        if not os.path.exists(dst):
            os.symlink(src, dst)
            linked += 1

print(f"Linked {linked} file references.")
print("\nVerifying — these should all print True:")
print(" transformer in /tmp:", os.path.exists(os.path.join(TMP_DIR, TRANSFORMER_FILE)))
print(" transformer in models:", os.path.exists(os.path.join(MODEL_DIR, TRANSFORMER_FILE)))
print(" gemma safetensors in models:", os.path.exists(
    os.path.join(MODEL_DIR, GEMMA4_SUBFOLDER, "gemma4-12b-ltx-v1_int8_convrot.safetensors")))
print(" gemma tokenizer.json in models:", os.path.exists(
    os.path.join(MODEL_DIR, GEMMA4_SUBFOLDER, "tokenizer.json")))

print("\nNow run the ORIGINAL script's 'Step 2' cell — it should print")
print("'✓ Already exists' for every file and download nothing.")


In [ ]:
#@title Step 2: Download Official LTX-2.5 22B Distilled Weights & Text Encoder
import os
import sys
import time
import json
import shutil

# Ensure clean huggingface_hub module state
for mod in list(sys.modules.keys()):
    if mod.startswith('huggingface_hub'):
        del sys.modules[mod]

from huggingface_hub import hf_hub_download

print('=== Downloading Official LTX-2.5 Distilled Weights ===')

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

TMP_DIR = '/tmp/models'
MODEL_DIR = os.path.abspath('Wan2GP/models')
CKPTS_DIR = os.path.abspath('Wan2GP/ckpts')
OUTPUTS_DIR = os.path.abspath('Wan2GP/outputs')
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CKPTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)

def safe_symlink(src, dest):
    if not os.path.exists(dest):
        try:
            os.symlink(src, dest)
        except Exception:
            pass

LTX25_REPO = 'DeepBeepMeep/LTX-2'

# 1. Official LTX-2.5 22B Distilled Diffusion Model (INT8 Convrot, ~19.5 GB) -> Download to /tmp/models
LTX25_DISTILLED_FILE = 'ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors'
dest_tf = os.path.join(MODEL_DIR, LTX25_DISTILLED_FILE)
dest_tf_ckpt = os.path.join(CKPTS_DIR, LTX25_DISTILLED_FILE)

if os.path.exists(dest_tf) and os.path.exists(dest_tf_ckpt):
    print(f'  ✓ Already exists: {LTX25_DISTILLED_FILE}')
else:
    print(f'Downloading LTX-2.5 Distilled INT8 weights ({LTX25_DISTILLED_FILE}, ~19.5 GB to temp storage)...')
    sys.stdout.flush()
    t0 = time.time()
    hf_hub_download(repo_id=LTX25_REPO, filename=LTX25_DISTILLED_FILE, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, LTX25_DISTILLED_FILE)
    shutil.rmtree(os.path.join(TMP_DIR, '.cache'), ignore_errors=True)
    if os.path.exists(actual):
        safe_symlink(actual, dest_tf)
        safe_symlink(actual, dest_tf_ckpt)
        elapsed = time.time() - t0
        print(f'  ✓ {LTX25_DISTILLED_FILE} ready! (Downloaded in {elapsed:.1f}s)')
        sys.stdout.flush()

# 2. Official LTX-2.5 Companion Components (Spatial Upscaler, VAEs, Connectors, Vocoder) -> Download to Wan2GP/models
LTX25_COMPANION_FILES = [
    'ltx-2.5-spatial-upscaler-x2-1.0_bf16.safetensors',
    'ltx-2.5-temporal-upscaler-x2-1.0_bf16.safetensors',
    'ltx-2.5-22b_video_embeddings_connector_int8_convrot.safetensors',
    'ltx-2.5-22b_audio_embeddings_connector_int8_convrot.safetensors',
    'ltx-2.5-22b_text_embedding_projection_bf16.safetensors',
    'ltx-2.5-22b_video_vae_bf16.safetensors',
    'ltx-2.5-22b_audio_vae_bf16.safetensors',
    'ltx-2.5-22b_vocoder_bf16.safetensors',
]

for f in LTX25_COMPANION_FILES:
    dest_m = os.path.join(MODEL_DIR, f)
    dest_c = os.path.join(CKPTS_DIR, f)
    if os.path.exists(dest_m) and os.path.exists(dest_c):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f}...')
    sys.stdout.flush()
    try:
        hf_hub_download(repo_id=LTX25_REPO, filename=f, local_dir=MODEL_DIR)
        shutil.rmtree(os.path.join(MODEL_DIR, '.cache'), ignore_errors=True)
        if os.path.exists(dest_m):
            safe_symlink(dest_m, dest_c)
            print(f'  ✓ {f} complete!')
    except Exception as e:
        print(f'  ⚠️ Warning downloading {f}: {e}')
    sys.stdout.flush()

# 3. LTX-2.5 Dedicated Gemma4 Text Encoder & Tokenizer Files (gemma4-12b-ltx-v1)
GEMMA4_SUBFOLDER = 'gemma4-12b-ltx-v1'
GEMMA4_DIR = os.path.join(MODEL_DIR, GEMMA4_SUBFOLDER)
GEMMA4_CKPT_DIR = os.path.join(CKPTS_DIR, GEMMA4_SUBFOLDER)
os.makedirs(GEMMA4_DIR, exist_ok=True)
os.makedirs(GEMMA4_CKPT_DIR, exist_ok=True)

GEMMA4_FILES = [
    'gemma4-12b-ltx-v1_int8_convrot.safetensors',
    'config.json',
    'chat_template.jinja',
    'tokenizer.json',
    'tokenizer_config.json'
]

print(f'Preparing LTX-2.5 Text Encoder ({GEMMA4_SUBFOLDER})...')
sys.stdout.flush()
for gf in GEMMA4_FILES:
    dest_gm = os.path.join(GEMMA4_DIR, gf)
    dest_gc = os.path.join(GEMMA4_CKPT_DIR, gf)
    if os.path.exists(dest_gm) and os.path.exists(dest_gc):
        continue
    print(f'Downloading {gf}...')
    sys.stdout.flush()
    try:
        hf_hub_download(
            repo_id=LTX25_REPO,
            subfolder=GEMMA4_SUBFOLDER,
            filename=gf,
            local_dir=MODEL_DIR
        )
        shutil.rmtree(os.path.join(MODEL_DIR, '.cache'), ignore_errors=True)
        # Handle nested subfolder if present
        nested = os.path.join(GEMMA4_DIR, GEMMA4_SUBFOLDER, gf)
        if os.path.exists(nested):
            shutil.move(nested, dest_gm)
        if os.path.exists(dest_gm):
            safe_symlink(dest_gm, dest_gc)
    except Exception as e:
        print(f'  Note on {gf}: {e}')
    sys.stdout.flush()

# Ensure all Gemma 4 files are symlinked to ckpts
for gf in GEMMA4_FILES:
    dest_gm = os.path.join(GEMMA4_DIR, gf)
    dest_gc = os.path.join(GEMMA4_CKPT_DIR, gf)
    if os.path.exists(dest_gm) and not os.path.exists(dest_gc):
        safe_symlink(dest_gm, dest_gc)

print('  ✓ LTX-2.5 Gemma-4 text encoder ready!')

# 4. Configure Wan2GP Defaults
DEFAULTS_DIR = os.path.abspath('Wan2GP/defaults')
distilled_default_config = {
    "model": {
        "name": "LTX-2 2.5 Distilled 22B",
        "architecture": "ltx2_25_22B",
        "description": "LTX-2.5 Distilled INT8 model optimized for fast 8-step single-stage inference on Dual GPUs.",
        "URLs": [
            "https://huggingface.co/DeepBeepMeep/LTX-2/resolve/main/ltx-2.5-22b-distilled_diffusion_model_bf16.safetensors",
            "https://huggingface.co/DeepBeepMeep/LTX-2/resolve/main/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"
        ],
        "preload_URLs": [],
        "ltx2_pipeline": "distilled"
    },
    "guidance_phases": 1,
    "num_inference_steps": 8,
    "video_length": 73,
    "resolution": "640x384"
}

# Align all default configs to Distilled INT8 weights
for config_file in ['ltx2_25_22B.json', 'ltx2_25_22B_distilled.json', 'ltx2_25_22B_distilled_nvfp4.json']:
    cfg_path = os.path.join(DEFAULTS_DIR, config_file)
    try:
        with open(cfg_path, 'w', encoding='utf-8') as f:
            json.dump(distilled_default_config, f, indent=4)
    except Exception as e:
        print(f'  Note writing {config_file}: {e}')

# Clean any remaining cache directories to maximize free disk & RAM
for d in ['/kaggle/working/.cache', os.path.join(MODEL_DIR, '.cache'), os.path.join(TMP_DIR, '.cache'), os.path.join(GEMMA4_DIR, '.cache')]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

print('\n✅ Official LTX-2.5 22B Distilled weights & environment prepared successfully!')

In [ ]:
#@title Step 3: Write the AIQUEST Academy High-Speed Streaming Web Application
import os
import sys

aiquest_app_code = """import gc
import os
import sys
import json
import time
import random
import tempfile
import glob
import traceback
import queue
import threading
import numpy as np
import subprocess
import psutil
import soundfile as sf
import torch
from PIL import Image

# ── Bootstrap Wan2GP ──
WAN2GP_DIR = os.path.abspath("Wan2GP")
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

# Wan2GP's own purpose-built loader for multi-frame video conditioning (decode +
# resize/crop + normalize to the model's latent-space convention). Reused as-is
# rather than hand-building the tensor, since it is what the rest of this
# codebase already relies on for exactly this.
from models.ltx2.ltx_pipelines.utils.media_io import load_video_conditioning

OUTPUTS_DIR = os.path.abspath(os.path.join(WAN2GP_DIR, "outputs"))
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)

# Enable PyTorch & cuDNN fast inference optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
# T4 (sm_75) has no FlashAttention kernel; mem-efficient SDPA is the fast path here.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# Clean any leftover lock files from previous runs
for lock_name in ['startup.lock', 'wgp.lock']:
    lock_path = os.path.join(WAN2GP_DIR, lock_name)
    if os.path.exists(lock_path):
        try:
            os.remove(lock_path)
        except Exception:
            pass

# Backward-compatibility patch for HfFolder in huggingface_hub
try:
    import huggingface_hub.utils as hf_utils
    if not hasattr(hf_utils, 'HfFolder'):
        class HfFolder:
            @staticmethod
            def get_token():
                import os
                try:
                    from huggingface_hub import get_token
                    return get_token() or os.environ.get('HF_TOKEN')
                except Exception:
                    return os.environ.get('HF_TOKEN')
            @staticmethod
            def save_token(token):
                pass
        setattr(hf_utils, 'HfFolder', HfFolder)
        if 'huggingface_hub.utils' in sys.modules:
            setattr(sys.modules['huggingface_hub.utils'], 'HfFolder', HfFolder)
except Exception:
    pass

# Patch ltx2.py in Wan2GP to disable requires_grad on empty models so int8 weights load cleanly
ltx2_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx2.py")
if os.path.isfile(ltx2_file):
    with open(ltx2_file, "r", encoding="utf-8") as f:
        ltx2_code = f.read()
    if 'def _load_component(model, path, sd_ops=None, postprocess=None, ignore_unused_weights=False):\\n            if postprocess is None and sd_ops is not None:' in ltx2_code:
        ltx2_code = ltx2_code.replace(
            'def _load_component(model, path, sd_ops=None, postprocess=None, ignore_unused_weights=False):\\n            if postprocess is None and sd_ops is not None:',
            'def _load_component(model, path, sd_ops=None, postprocess=None, ignore_unused_weights=False):\\n            model.requires_grad_(False)\\n            if postprocess is None and sd_ops is not None:'
        )
        with open(ltx2_file, "w", encoding="utf-8") as f:
            f.write(ltx2_code)

# Patch distilled.py in Wan2GP to use model dtype instead of hardcoded bfloat16 (enables fast FP16 on T4)
distilled_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_pipelines", "distilled.py")
if os.path.isfile(distilled_file):
    with open(distilled_file, "r", encoding="utf-8") as f:
        dist_code = f.read()
    dist_code = dist_code.replace("self.dtype = torch.bfloat16", "self.dtype = getattr(models, 'dtype', torch.float16)")
    dist_code = dist_code.replace("dtype = torch.bfloat16", "dtype = getattr(self, 'dtype', torch.float16)")
    with open(distilled_file, "w", encoding="utf-8") as f:
        f.write(dist_code)

# Patch CausalConv3d in Wan2GP video_vae to prevent dtype mismatch between latents and VAE conv weights
conv_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "video_vae", "convolution.py")
if os.path.isfile(conv_file):
    with open(conv_file, "r", encoding="utf-8") as f:
        conv_code = f.read()
    if "x = self.conv(x)" in conv_code and "x.dtype != self.conv.weight.dtype" not in conv_code:
        conv_code = conv_code.replace(
            "        x = self.conv(x)",
            "        if hasattr(self, 'conv') and hasattr(self.conv, 'weight') and x.dtype != self.conv.weight.dtype:\\n            x = x.to(dtype=self.conv.weight.dtype)\\n        x = self.conv(x)"
        )
        with open(conv_file, "w", encoding="utf-8") as f:
            f.write(conv_code)

# Patch audio_vae.py in Wan2GP to auto-cast latents to audio decoder and vocoder weight dtypes
audio_vae_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "audio_vae", "audio_vae.py")
if os.path.isfile(audio_vae_file):
    with open(audio_vae_file, "r", encoding="utf-8") as f:
        audio_vae_code = f.read()
    if "decoded_audio = audio_decoder(latent)" in audio_vae_code and "latent.to(dtype=" not in audio_vae_code:
        audio_vae_code = audio_vae_code.replace(
            "    decoded_audio = audio_decoder(latent)",
            "    if hasattr(audio_decoder, 'conv_in') and hasattr(audio_decoder.conv_in, 'weight'):\\n        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\\n    decoded_audio = audio_decoder(latent)\\n    if hasattr(vocoder, 'vocoder') and hasattr(vocoder.vocoder, 'conv_pre'):\\n        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)"
        )
        with open(audio_vae_file, "w", encoding="utf-8") as f:
            f.write(audio_vae_code)

# Patch vocoder.py in Wan2GP to prevent buffer dtype mismatch in LowPassFilter1d and STFT
vocoder_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "audio_vae", "vocoder.py")
if os.path.isfile(vocoder_file):
    with open(vocoder_file, "r", encoding="utf-8") as f:
        vocoder_code = f.read()
    vocoder_code = vocoder_code.replace(
        "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)",
        "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\\n        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)"
    )
    vocoder_code = vocoder_code.replace(
        "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
        "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\\n        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)"
    )
    with open(vocoder_file, "w", encoding="utf-8") as f:
        f.write(vocoder_code)

# Patch PixelNorm in Wan2GP to compute RMS in float32 (fixes FP16 overflow into flat gray background)
norm_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "model", "common", "normalization.py")
if os.path.isfile(norm_file):
    with open(norm_file, "r", encoding="utf-8") as f:
        norm_code = f.read()
    if "x_float = x.float()" not in norm_code:
        norm_code = norm_code.replace(
            "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\\n        # Normalize by the root-mean-square (RMS).\\n        rms = torch.sqrt(mean_sq + self.eps)\\n        return x / rms",
            "        orig_dtype = x.dtype\\n        x_float = x.float()\\n        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\\n        rms = torch.sqrt(mean_sq + self.eps)\\n        return (x_float / rms).to(orig_dtype)"
        )
        with open(norm_file, "w", encoding="utf-8") as f:
            f.write(norm_code)

# Patch feature_extractor.py in Wan2GP to compute token RMS in float32
fe_file = os.path.join(WAN2GP_DIR, "models", "ltx2", "ltx_core", "text_encoders", "gemma", "feature_extractor.py")
if os.path.isfile(fe_file):
    with open(fe_file, "r", encoding="utf-8") as f:
        fe_code = f.read()
    if "enc_f = encoded_text.float()" not in fe_code:
        fe_code = fe_code.replace(
            "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\\n    b, t, d, l = encoded_text.shape\\n    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\\n    normed = encoded_text * torch.rsqrt(variance + 1e-6)\\n    normed = normed.reshape(b, t, d * l)\\n    mask_3d = attention_mask.bool().unsqueeze(-1)\\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
            "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\\n    orig_dtype = encoded_text.dtype\\n    enc_f = encoded_text.float()\\n    b, t, d, l = enc_f.shape\\n    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\\n    normed = enc_f * torch.rsqrt(variance + 1e-6)\\n    normed = normed.reshape(b, t, d * l).to(orig_dtype)\\n    mask_3d = attention_mask.bool().unsqueeze(-1)\\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))"
        )
        with open(fe_file, "w", encoding="utf-8") as f:
            f.write(fe_code)

import gradio as gr
from shared.utils.audio_video import save_video
from mmgp import offload, quant_router
from models.ltx2.ltx2_handler import family_handler

# Register Wan2GP custom quantization handlers (INT8 ConvRot, FP8, NVFP4, GGUF) with mmgp
_HANDLER_MODULES = [
    "shared.qtypes.scaled_fp8",
    "shared.qtypes.nvfp4",
    "shared.qtypes.bnb_nf4",
    "shared.qtypes.nunchaku_int4",
    "shared.qtypes.nunchaku_fp4",
    "shared.qtypes.asym_w4a8_int8",
    "shared.qtypes.int8_convrot",
    "shared.qtypes.gguf",
]
quant_router.unregister_handler(".fp8_quanto_bridge")
for handler in _HANDLER_MODULES:
    quant_router.register_handler(handler)
from shared.qtypes import gguf as gguf_handler
quant_router.register_file_extension("gguf", gguf_handler)

# ==== GPU INFO ====
gpu_count = torch.cuda.device_count()
print(f"GPUs Available: {gpu_count}")
for i in range(gpu_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB VRAM)")

ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

# ==== LOAD LTX-2.5 22B DISTILLED MODEL IN-MEMORY (HIGH-SPEED BUDGET ARCHITECTURE) ====
print("\\n=== Loading LTX-2.5 22B Distilled Model in Memory (One-Time Startup) ===")
sys.stdout.flush()

base_model_type = "ltx2_25_22B"
model_def = {
    "ltx2_pipeline": "distilled",
    "name": "LTX-2 2.5 Distilled 22B",
    "architecture": "ltx2_25_22B"
}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

# Auto-resolve and verify text encoder and transformer paths
for base_path in ["models", "ckpts"]:
    nested_gemma = os.path.join(base_path, "gemma4-12b-ltx-v1", "gemma4-12b-ltx-v1")
    if os.path.isdir(nested_gemma):
        for f in os.listdir(nested_gemma):
            dst = os.path.join(base_path, "gemma4-12b-ltx-v1", f)
            if not os.path.exists(dst):
                try:
                    os.replace(os.path.join(nested_gemma, f), dst)
                except Exception:
                    pass

# Ensure all symlinks exist between /tmp/models, models/, and ckpts/
for d_from in ["/tmp/models", "models"]:
    if os.path.exists(d_from):
        for f in os.listdir(d_from):
            if f.startswith("."): continue
            src = os.path.join(d_from, f)
            for d_to in ["models", "ckpts"]:
                dst = os.path.join(d_to, f)
                if not os.path.exists(dst):
                    try:
                        os.symlink(os.path.abspath(src), dst)
                    except Exception:
                        pass

# Specifically link gemma4-12b-ltx-v1 folder contents
for gf in ["gemma4-12b-ltx-v1_int8_convrot.safetensors", "config.json", "chat_template.jinja", "tokenizer.json", "tokenizer_config.json"]:
    src_gm = os.path.join("models", "gemma4-12b-ltx-v1", gf)
    dst_gc = os.path.join("ckpts", "gemma4-12b-ltx-v1", gf)
    os.makedirs(os.path.dirname(dst_gc), exist_ok=True)
    if os.path.exists(src_gm) and not os.path.exists(dst_gc):
        try:
            os.symlink(os.path.abspath(src_gm), dst_gc)
        except Exception:
            pass

text_encoder_file = "ckpts/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors"
if not os.path.exists(text_encoder_file):
    text_encoder_file = "models/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors"

transformer_path = "models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"
if not os.path.exists(transformer_path):
    transformer_path = "ckpts/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"
if not os.path.exists(transformer_path) and os.path.exists("/tmp/models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"):
    transformer_path = "/tmp/models/ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors"

print(f"  Transformer : {os.path.basename(transformer_path)}")
print(f"  Text Encoder: {os.path.basename(text_encoder_file)}")
sys.stdout.flush()

MODEL_DTYPE = torch.float16
VAE_DTYPE   = torch.float16

# Loaded under no_grad rather than inference_mode: inference_mode marks every weight
# as an "inference tensor", which then blows up the moment it is touched from the
# generation worker thread (grad mode is thread-local and defaults to ON there).
# The video VAE *encoder* is only ever exercised by Image-to-Video, so that failure
# mode showed up as "T2V works, I2V doesn't".
with torch.no_grad():
    with torch.set_grad_enabled(False):
        wan_model, pipe = family_handler.load_model(
            model_filename=transformer_path,
            model_type="ltx2_25_22B_distilled",
            base_model_type=base_model_type,
            model_def=model_def,
            dtype=MODEL_DTYPE,
            VAE_dtype=VAE_DTYPE,
            text_encoder_filename=text_encoder_file,
        )

kwargs = {}
if isinstance(pipe, dict) and "pipe" in pipe:
    kwargs = pipe
    pipe = kwargs.pop("pipe")

loras = kwargs.pop("loras", [])

if "transformer" in pipe and "transformer" not in loras:
    loras.append("transformer")

# The INT8 transformer is ~19.5 GB and cannot fit in a 15 GB T4, so whatever does not
# stay resident is streamed from pinned RAM on *every* denoising step. That PCIe traffic
# - not the matmuls - dominates step time here, so size the resident slice from the VRAM
# that is actually free rather than pinning it at a conservative 6000 MB.
_free_mb0 = int(torch.cuda.mem_get_info(0)[0] / 1024**2)
TRANSFORMER_BUDGET_MB = int(max(6000, min(10500, _free_mb0 - 4500)))

# Same bottleneck applies to the Gemma-12B INT8 text encoder (~12 GB): at the old
# 1500 MB budget it streamed ~10 GB per prompt, costing ~48 s on every new prompt.
# It is evicted before the transformer loads, so the two budgets never co-reside.
TEXT_ENCODER_BUDGET_MB = int(max(1500, min(10000, _free_mb0 - 6000)))

print(f"\\nApplying mmgp Profile 4 with Partial Pinning & Async DMA Transfers...")
print(f"  Free VRAM on cuda:0: {_free_mb0} MB -> transformer {TRANSFORMER_BUDGET_MB} MB, text encoder {TEXT_ENCODER_BUDGET_MB} MB")
sys.stdout.flush()

# THE Image-to-Video FIX.
# The LTX video VAE encoder overflows in FP16 and returns an all-NaN conditioning
# latent (measured: 57344 of 57344 values NaN). That NaN is injected into latent
# frame 0 and spreads through joint video/audio attention, which is what produced a
# black clip with garbage audio and no traceback. The decoder is fine in FP16 - its
# input is an already-normalised latent - which is why Text-to-Video never hit this.
# Promote ONLY the encoder to FP32 and mark it with mmgp's `_lock_dtype` so
# convertWeightsFloatTo cannot fold it back to FP16. Costs ~0.7 GB VRAM while it is
# resident, and it runs on a single frame per generation.
try:
    _video_encoder = pipe.get("video_encoder")
    if _video_encoder is not None:
        _video_encoder.float()
        for _m in _video_encoder.modules():
            # mmgp reads _lock_dtype as a torch.dtype: change_dtype() skips locked
            # submodules, profile() leaves their weights alone, and hooked modules get
            # their float32 inputs kept at float32. It must be a dtype - a bool here is
            # fed straight into tensor.to() and silently turns activations into Bool.
            _m._lock_dtype = torch.float32
        print("  ✓ Video VAE encoder pinned to FP32 (fixes all-NaN I2V conditioning latent)")
    else:
        print("  ⚠️ No 'video_encoder' in pipe - Image-to-Video may still produce NaN latents")
except Exception as _e:
    print(f"  ⚠️ Could not promote video encoder to FP32: {_e}")
sys.stdout.flush()

offload.profile(
    pipe,
    profile_no=4,
    pinnedMemory="transformer",
    partialPinning=True,
    perc_reserved_mem_max=0.35,
    asyncTransfers=True,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    loras=loras,
    budgets={
        # Sized above from free VRAM: more resident weights == less per-step streaming.
        "transformer":       TRANSFORMER_BUDGET_MB,
        "text_encoder":      TEXT_ENCODER_BUDGET_MB,
        # FP32 encoder is 2x the FP16 footprint; give it room.
        "video_encoder":     3000,
        "video_decoder":     3000,
        "audio_encoder":     1000,
        "audio_decoder":     1000,
        "vocoder":           500,
        "spatial_upsampler": 1500,
        "vae":               1000,
        "*":                 1000,
    },
    **kwargs
)

offload.shared_state["_attention"] = "sdpa"
offload.shared_state["_radial"] = False

print("\\n✅ Setup complete! LTX-2.5 22B Distilled Model loaded & pinned in memory permanently.")
sys.stdout.flush()

# ==== RESOLUTION PRESETS ====
def snap64(value):
    # LTX-2 rounds width/height UP to a multiple of 64 internally before it builds the
    # latent grid. Snapping here as well keeps the size we prepare the start frame for
    # identical to the size the pipeline actually generates at.
    return max(64, int(round(float(value) / 64.0)) * 64)

def fit_to_image(image, width, height):
    # Re-derive WxH from the uploaded frame's own aspect ratio while keeping the
    # preset's pixel budget, so an Image-to-Video run is not cropped to the preset ratio.
    img_w, img_h = image.size
    scale = ((width * height) / float(img_w * img_h)) ** 0.5
    return snap64(img_w * scale), snap64(img_h * scale)

def prepare_conditioning_image(image, width, height):
    # Centre-crop to the target ratio, then resample to exactly WxH.
    img_w, img_h = image.size
    target_ratio = width / float(height)
    src_ratio    = img_w / float(img_h)
    if src_ratio > target_ratio:
        new_w = int(round(img_h * target_ratio))
        left  = (img_w - new_w) // 2
        image = image.crop((left, 0, left + new_w, img_h))
    elif src_ratio < target_ratio:
        new_h = int(round(img_w / target_ratio))
        top   = (img_h - new_h) // 2
        image = image.crop((0, top, img_w, top + new_h))
    return image.resize((width, height), Image.Resampling.LANCZOS)

def get_vae_tile_size(width, height):
    # Spatial tiling off at low resolution - it is what caused the 256 px seam
    # artefacts - but a full-frame decode of a 1080p clip will not fit on a 16 GB T4,
    # so tile above ~704p. Single-frame I2V conditioning encodes are unaffected:
    # encode_video() drops spatial tiling whenever the input is one frame.
    longest = max(int(width), int(height))
    if longest <= 1024:
        return 0
    if longest <= 1536:
        return 512
    return 256

def get_resolution(preset_str, aspect_ratio_str):
    base_resolutions = {
        "Fast Preview (384p - ~1-2 min)": 384,
        "Balanced (480p - ~3-5 min)": 480,
        "High Quality (704p - ~6-8 min)": 704,
        "Cinema 1080p (1088p - High Detail)": 1088,
    }
    ratios = {
        "16:9 Landscape": 16/9, "4:3 Standard": 4/3,
        "1:1 Square": 1.0, "3:4 Portrait": 3/4, "9:16 Portrait": 9/16,
    }
    base = base_resolutions.get(preset_str, 384)
    ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        height = base
        width = int(base * ratio)
    else:
        width = base
        height = int(base / ratio)
    return snap64(width), snap64(height)

# ==== DENOISING STEP CONTROL ====
# The distilled pipeline ignores `sampling_steps` entirely: it always walks the
# hardcoded 9-value DISTILLED_SIGMA_VALUES schedule, i.e. exactly 8 steps. Rebinding
# that module global is the only way to actually trade denoising steps for speed.
try:
    from models.ltx2.ltx_pipelines import distilled as _ltx_distilled
    FULL_SIGMAS = list(_ltx_distilled.DISTILLED_SIGMA_VALUES)
except Exception as _e:
    # Upstream moved/renamed the schedule: fall back to the fixed 8 steps.
    _ltx_distilled = None
    FULL_SIGMAS = [1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0]
    print(f"  ⚠️ Denoising-step override unavailable ({_e}); steps locked at 8.")

# The first five sigmas (1.0 -> 0.975) barely move the latent; shorter schedules drop
# from that cluster first and keep the three sigmas that do the real denoising work.
STEP_SCHEDULES = {
    8: FULL_SIGMAS,
    7: [1.0, 0.99375, 0.9875, 0.975, 0.909375, 0.725, 0.421875, 0.0],
    6: [1.0, 0.9875, 0.975, 0.909375, 0.725, 0.421875, 0.0],
    5: [1.0, 0.975, 0.909375, 0.725, 0.421875, 0.0],
    4: [1.0, 0.909375, 0.725, 0.421875, 0.0],
}

def set_denoising_steps(num_steps):
    schedule = STEP_SCHEDULES.get(int(num_steps), FULL_SIGMAS)
    if _ltx_distilled is not None:
        _ltx_distilled.DISTILLED_SIGMA_VALUES = schedule
        return len(schedule) - 1
    return len(FULL_SIGMAS) - 1

# ==== IMAGE-TO-VIDEO CONDITIONING: DIAGNOSTICS + HARDENING ====
# The video VAE ships as BF16 and loads as BF16 (mmgp only converts float32 weights),
# while the pipeline computes in FP16. image_conditionings_* returns the encoded
# conditioning latent WITHOUT casting it to the pipeline dtype - unlike editanything.py,
# which does cast. A non-finite or wrongly-typed conditioning latent is injected into
# frame 0 and then spreads through joint video/audio attention, which is what produces
# a fully black video with garbage audio and no traceback.
# So: report the numbers, cast to the pipeline dtype, and scrub non-finite values.
import models.ltx2.ltx_pipelines.utils.helpers as _ltx_helpers

LTX_DIAG = True

def _diag_tensor(tag, t, max_elems=1000000):
    # Strides over very large tensors instead of materialising a float32 copy of them:
    # a full copy of a decoded 121-frame clip is enough to get the process OOM-killed.
    if not torch.is_tensor(t):
        print(f"  [diag] {tag}: not a tensor ({type(t)})")
        sys.stdout.flush()
        return 0
    flat = t.detach().reshape(-1)
    total = flat.numel()
    if total > max_elems:
        stride = (total + max_elems - 1) // max_elems
        sample = flat[::stride].float()
        note = f" [sampled 1/{stride}]"
    else:
        sample = flat.float()
        note = ""
    n_nan = int(torch.isnan(sample).sum())
    n_inf = int(torch.isinf(sample).sum())
    finite_mask = torch.isfinite(sample)
    if bool(finite_mask.any()):
        finite = sample[finite_mask]
        lo, hi, mu = float(finite.min()), float(finite.max()), float(finite.mean())
        del finite
    else:
        lo = hi = mu = float("nan")
    del sample, finite_mask
    print(f"  [diag] {tag}: shape={tuple(t.shape)} dtype={t.dtype} dev={t.device} "
          f"min={lo:.4g} max={hi:.4g} mean={mu:.4g} nan={n_nan} inf={n_inf}{note}")
    sys.stdout.flush()
    return n_nan + n_inf

def _encoder_dtype(encoder):
    try:
        return next(encoder.parameters()).dtype
    except Exception:
        return None

def _wrap_conditioner(fn, label):
    def inner(*args, **kwargs):
        pipe_dtype = kwargs.get("dtype")
        images = kwargs.get("images")
        enc_dtype = _encoder_dtype(kwargs.get("video_encoder"))
        # Encode at the encoder's own precision (FP32, pinned above) rather than at the
        # pipeline's FP16, then hand the latent back in the pipeline dtype.
        if enc_dtype is not None and enc_dtype != pipe_dtype:
            kwargs = dict(kwargs)
            kwargs["dtype"] = enc_dtype
        if LTX_DIAG:
            print(f"  [diag] {label}: {len(images or [])} item(s) -> "
                  f"{kwargs.get('width')}x{kwargs.get('height')}, pipeline dtype={pipe_dtype}, "
                  f"encoder dtype={enc_dtype}, tiling={kwargs.get('tiling_config') is not None}")
            sys.stdout.flush()
        items = fn(*args, **kwargs)

        # Last-resort safety net: if the latent still came back non-finite, the FP32 pin
        # did not take (e.g. mmgp reloaded FP16 weights). Redo the encode on CPU in FP32,
        # which cannot overflow. Slow, but only ever runs when the fast path failed.
        def _nonfinite(items_):
            for it in items_:
                a = "latent" if hasattr(it, "latent") else ("keyframes" if hasattr(it, "keyframes") else None)
                if a is not None:
                    v = getattr(it, a)
                    if torch.is_tensor(v) and not bool(torch.isfinite(v).all()):
                        return True
            return False

        encoder = kwargs.get("video_encoder")
        if encoder is not None and _nonfinite(items):
            print("  [diag] ⚠️ conditioning latent still non-finite - retrying encode on CPU in FP32")
            sys.stdout.flush()
            try:
                origin = next(encoder.parameters()).device
                encoder.to(device="cpu", dtype=torch.float32)
                cpu_kwargs = dict(kwargs)
                cpu_kwargs["dtype"] = torch.float32
                cpu_kwargs["device"] = torch.device("cpu")
                cpu_items = fn(*args, **cpu_kwargs)
                encoder.to(device=origin)
                target_device = kwargs.get("device")
                for it in cpu_items:
                    a = "latent" if hasattr(it, "latent") else ("keyframes" if hasattr(it, "keyframes") else None)
                    if a is not None and target_device is not None:
                        setattr(it, a, getattr(it, a).to(device=target_device))
                items = cpu_items
                print("  [diag] ✓ CPU FP32 re-encode succeeded")
                sys.stdout.flush()
            except Exception as retry_error:
                print(f"  [diag] ⚠️ CPU FP32 re-encode failed ({retry_error}); scrubbing instead")
                sys.stdout.flush()
                try:
                    encoder.to(device=origin)
                except Exception:
                    pass

        dtype = pipe_dtype
        for i, item in enumerate(items):
            attr = "latent" if hasattr(item, "latent") else ("keyframes" if hasattr(item, "keyframes") else None)
            if attr is None:
                continue
            lat = getattr(item, attr)
            if LTX_DIAG:
                _diag_tensor(f"{label}[{i}].{attr}", lat)
            # Kept independent of LTX_DIAG on purpose: this scrub is the safety net that
            # stops a non-finite conditioning latent from blackening the whole clip, so
            # turning the logging off must not turn it off too.
            fixed = lat
            if torch.is_tensor(lat) and not bool(torch.isfinite(lat).all()):
                fixed = torch.nan_to_num(lat.float(), nan=0.0, posinf=0.0, neginf=0.0)
                print(f"  ⚠️ scrubbed non-finite values from {label}[{i}]")
                sys.stdout.flush()
            if dtype is not None and fixed.dtype != dtype:
                if LTX_DIAG:
                    print(f"  [diag] casting {label}[{i}] {fixed.dtype} -> {dtype} (pipeline dtype)")
                    sys.stdout.flush()
                fixed = fixed.to(dtype)
            if fixed is not lat:
                setattr(item, attr, fixed)
        return items
    return inner

for _fname, _label in (("image_conditionings_by_replacing_latent", "start_frame_cond"),
                       ("image_conditionings_by_adding_guiding_latent", "end_frame_cond")):
    _orig = getattr(_ltx_helpers, _fname, None)
    if _orig is not None:
        _wrapped = _wrap_conditioner(_orig, _label)
        setattr(_ltx_helpers, _fname, _wrapped)
        # distilled.py imported these by value, so rebind them there too.
        if _ltx_distilled is not None and hasattr(_ltx_distilled, _fname):
            setattr(_ltx_distilled, _fname, _wrapped)
        print(f"  ✓ instrumented {_fname}")
sys.stdout.flush()


# ==== REAL-TIME STREAMING VIDEO GENERATION (TIMEOUT-FREE) ====
def Video_Generation(prompt, input_image_start, input_image_end, seed, duration_dropdown,
                     resolution_dropdown, aspect_ratio_dropdown,
                     guide_scale=3.0, audio_cfg=7.0, num_steps=8, input_video_strength=1.0,
                     tail_clip=None, image_mode=False, input_ref_images=None):
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        offload.shared_state["_attention"] = "sdpa"
        offload.shared_state["_radial"] = False

        duration_map = {
            "2 Seconds (49 frames - Fast)": 49,
            "3 Seconds (73 frames - Standard)": 73,
            "5 Seconds (121 frames - Long)": 121,
            "8 Seconds (193 frames)": 193,
            "10 Seconds (241 frames)": 241,
        }
        frame_rate = 24.0
        num_frames = duration_map.get(duration_dropdown, 73)

        # Reference-image generation (image_mode=True): truncates the output to a
        # single frame regardless of frame count, so there is no reason to pay for
        # more than the minimum the model will actually accept. 9 = latent_stride(8)+1
        # is the smallest valid frame count for this pipeline - confirmed directly
        # against the model's own enforced floor, not a guess. This does NOT skip
        # the model's internal upscale/refine pass (that flag exists but contradicts
        # observed behavior in every production run so far, so it is deliberately
        # not used here) - it only avoids paying for extra frames we'd discard anyway.
        if image_mode:
            num_frames = 9
            print(f"  [ref-image] image_mode active - forcing num_frames to the minimum "
                  f"valid value (9) regardless of duration_dropdown, since only 1 frame "
                  f"is kept either way.")

        # Reference-image conditioning for a normal scene: "I" is the letters-filter
        # flag this pipeline's generate() checks for input_ref_images to be read at
        # all (confirmed in ltx2_handler.py's own declared filter for this exact
        # checkpoint) - without it, a supplied input_ref_images would be silently
        # ignored rather than raising, which is worse than an explicit flag.
        video_prompt_type = "I" if input_ref_images else ""

        width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown)

        if seed is None or seed < 0:
            seed = random.randint(0, 2**32 - 1)
        seed = int(seed)

        image_start = None
        image_end   = None
        if input_image_start is not None:
            image_start = Image.open(input_image_start).convert("RGB")
        if input_image_end is not None:
            image_end = Image.open(input_image_end).convert("RGB")

        # ---- Image-to-Video conditioning ----
        # LTX-2 encodes the conditioning frame into a latent that must match the target
        # latent grid exactly, and it centre-crops anything that does not. Re-derive the
        # output size from the uploaded frame and resample the frame to it, so the first
        # frame of the video is the image the user actually supplied.
        ref_image = image_start if image_start is not None else image_end
        if ref_image is not None:
            width, height = fit_to_image(ref_image, width, height)
            if image_start is not None:
                image_start = prepare_conditioning_image(image_start, width, height)
            if image_end is not None:
                image_end = prepare_conditioning_image(image_end, width, height)

        # ---- Tail-clip motion continuity ----
        # A single conditioning frame (image_start, above) carries position/appearance
        # at one instant but no velocity - the model has no way to tell which direction
        # anything was already moving. Feeding a short multi-frame clip from the tail of
        # the previous scene as `input_video` gives it real inter-frame motion to continue,
        # not just a static image plus a text guess. This REPLACES image_start for this
        # call when present (both targeting frame 0 at once is not a combination this
        # pipeline is designed for) - image_start remains the fallback path when no
        # tail clip was supplied (fresh scenes, or chaining that fell back to a single
        # extracted frame upstream).
        input_video = None
        prefix_frames_count = 0
        tail_clip_frame_count = 0
        if tail_clip is not None:
            try:
                input_video = load_video_conditioning(
                    video_path=tail_clip, height=height, width=width,
                    # frame_cap is a REQUIRED int in this Wan2GP build - it's used as a
                    # per-frame decrement during decoding (frame_cap -= 1, stop at 0),
                    # not an optional "no limit" style cap, so None is not valid here.
                    # 128 is comfortably above any realistic tail clip at typical frame
                    # rates even if TAIL_CLIP_SECONDS is increased later.
                    frame_cap=128, dtype=MODEL_DTYPE, device=torch.device("cuda:0"),
                )
                if input_video.ndim == 5:  # [1, C, T, H, W] -> [C, T, H, W]
                    input_video = input_video.squeeze(0)
                tail_clip_frame_count = int(input_video.shape[1])
                prefix_frames_count = tail_clip_frame_count
                image_start = None  # tail clip supersedes the single-frame start
                print(f"  [tail-clip] loaded {tail_clip_frame_count} frames from {tail_clip}, "
                      f"tensor shape={tuple(input_video.shape)} dtype={input_video.dtype}")
            except Exception as tail_clip_err:
                print(f"  [tail-clip] FAILED to load ({tail_clip_err}) - "
                      f"falling back to whatever single-frame conditioning was provided.")
                input_video = None
                prefix_frames_count = 0
            sys.stdout.flush()

        num_steps = set_denoising_steps(num_steps)

        free_vram = torch.cuda.mem_get_info()[0] / 1024**3
        ram = psutil.virtual_memory()
        if image_mode:
            mode = "REF-IMAGE"
        elif input_video is not None:
            mode = f"I2V tail-clip ({tail_clip_frame_count}f)"
        elif image_start is None:
            mode = "T2V"
        else:
            mode = "I2V first+last" if image_end else "I2V start"

        if input_ref_images:
            # Reference-conditioning and tail-clip/image_start are structurally
            # independent in this model (confirmed in source: they write to
            # separate conditioning lists) - both can be active on the same scene,
            # so the banner reflects both rather than only the continuity mode.
            mode = f"{mode} +ref"

        print(f"\\n{'='*60}")
        print(f"🎬 Generating LTX-2.5 [{mode}]: {width}x{height}, {num_frames} frames (~{num_frames/24.0:.1f}s), seed={seed}")
        print(f"  VRAM free: {free_vram:.2f} GB | RAM free: {ram.available / 1024**3:.1f} GB")
        print(f"  Prompt: {prompt[:120]}{'...' if len(prompt) > 120 else ''}")
        print(f"  Video CFG: {guide_scale} | Audio CFG: {audio_cfg} | Steps: {num_steps} | "
              f"Continuity input: {mode}")
        print(f"{'='*60}")
        sys.stdout.flush()

        yield None, f"⏳ Initializing LTX-2.5 [{mode}] ({width}x{height}, {num_frames} frames, seed: {seed})..."

        # Progress queue for real-time WebSocket updates across threads
        msg_queue = queue.Queue()
        total_steps = [int(num_steps)]
        current_step = [0]
        step_times = []
        last_step_time = [time.time()]
        # The text encoder drives the same callback before denoising starts, so its
        # progress was being reported as "Denoising step n/48". Track the phase and
        # label each one for what it actually is.
        phase = ["🔤 Text encode"]

        # One redrawn line instead of a print per step. Carriage-return rewrite, capped
        # at 4 redraws/sec so a fast phase cannot flood the log either.
        bar = {"active": False, "last": 0.0}

        def _draw_bar(label, cur, total, dt, rem_sec, free_v, force=False):
            now = time.time()
            if not force and now - bar["last"] < 0.25:
                return
            bar["last"] = now
            total = max(1, int(total))
            cur = max(0, min(int(cur), total))
            width = 22
            filled = int(width * cur / total)
            meter = "█" * filled + "░" * (width - filled)
            pct = 100.0 * cur / total
            sys.stdout.write(
                f"\\r  {label} |{meter}| {cur}/{total} ({pct:3.0f}%) · "
                f"{dt:.1f}s/step · ~{rem_sec:.0f}s left · VRAM {free_v:.1f} GB free    "
            )
            sys.stdout.flush()
            bar["active"] = True

        def _end_bar():
            if bar["active"]:
                sys.stdout.write("\\n")
                sys.stdout.flush()
                bar["active"] = False

        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            if is_start:
                if override_num_inference_steps is not None:
                    total_steps[0] = override_num_inference_steps
                current_step[0] = 0
                last_step_time[0] = time.time()
                return
            now = time.time()
            dt = now - last_step_time[0]
            last_step_time[0] = now
            step_times.append(dt)
            current_step[0] += 1
            if LTX_DIAG and current_step[0] == 1 and latent is not None:
                _end_bar()
                # If the latent is already poisoned one step in, the conditioning
                # latent is the source; if it is clean here, the denoising loop is.
                _diag_tensor("latent after step 1", latent)
            free_v = torch.cuda.mem_get_info()[0] / 1024**3
            avg_dt = sum(step_times) / len(step_times)
            rem_steps = max(total_steps[0] - current_step[0], 0)
            rem_sec = rem_steps * avg_dt
            msg = f"{phase[0]} {current_step[0]}/{total_steps[0]} ({dt:.1f}s/step | ~{rem_sec:.0f}s left | VRAM: {free_v:.1f} GB free)"
            _draw_bar(phase[0], current_step[0], total_steps[0], dt, rem_sec, free_v,
                      force=current_step[0] >= total_steps[0])
            msg_queue.put(msg)

        _stage_labels = {
            "VAE Encoding": "🎞️ VAE Encoding input frames...",
            "VAE Decoding": "🎬 VAE Decoding latents → video frames...",
            "Upsampling":   "🔭 Spatial upsampling latents...",
        }

        def set_progress_status(status: str):
            # Everything after text encoding belongs to the denoising phase.
            if phase[0] != "⚡ Denoise":
                phase[0] = "⚡ Denoise"
                step_times.clear()
                current_step[0] = 0
                last_step_time[0] = time.time()
            _end_bar()
            label = _stage_labels.get(status, f"⏳ {status}...")
            print(f"  [{status}] {label}")
            sys.stdout.flush()
            msg_queue.put(label)
            if "Decoding" in status or "Upsampling" in status:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        gen_kwargs = dict(
            input_prompt=prompt,
            image_start=image_start,
            height=height,
            width=width,
            frame_num=num_frames,
            fps=frame_rate,
            seed=seed,
            callback=cb,
            VAE_tile_size=get_vae_tile_size(width, height),
            input_video=input_video,
            prefix_frames_count=int(prefix_frames_count),
            input_video_strength=float(input_video_strength),
            denoising_strength=1.0,
            image_mode=int(bool(image_mode)),
            input_ref_images=input_ref_images,
            video_prompt_type=video_prompt_type,
            guide_scale=float(guide_scale),
            audio_cfg_scale=float(audio_cfg),
            sampling_steps=int(num_steps),
            guide_phases=1,
            sample_solver="distilled_8_steps",
            self_refiner_setting=0,
            n_prompt="",
            set_progress_status=set_progress_status,
        )
        if image_end is not None:
            gen_kwargs["image_end"] = image_end

        result_holder = {}
        error_holder = {}

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        def run_generation():
            # Grad mode is thread-local and defaults to ON in a freshly spawned thread,
            # so it has to be turned off here and not only around the model load.
            try:
                with torch.no_grad():
                    t0 = time.time()
                    res = wan_model.generate(**gen_kwargs)
                    result_holder["result"] = res
                    result_holder["elapsed"] = time.time() - t0
            except Exception as exc:
                error_holder["error"] = exc
                traceback.print_exc()

        gen_thread = threading.Thread(target=run_generation)
        gen_thread.start()

        # Stream periodic WebSocket heartbeats while generation is in progress
        last_yield_time = time.time()
        current_status = "⏳ Starting in-memory single-stage 8-step generation..."
        while gen_thread.is_alive():
            # Check for new step messages
            updated = False
            while not msg_queue.empty():
                current_status = msg_queue.get_nowait()
                updated = True

            # Yield heartbeat every 2 seconds to keep WebSocket proxy alive
            now = time.time()
            if updated or (now - last_yield_time >= 2.0):
                last_yield_time = now
                yield None, current_status
            time.sleep(0.5)

        gen_thread.join()
        _end_bar()

        if "error" in error_holder:
            raise error_holder["error"]

        result = result_holder.get("result")
        elapsed = result_holder.get("elapsed", 0)
        print(f"  [Pipeline Completed] Finished in {elapsed:.1f}s")
        sys.stdout.flush()

        yield None, f"🎬 Generation finished in {elapsed:.1f}s! Saving MP4 and muxing audio..."

        if result is None:
            yield None, "❌ Generation returned no output."
            return

        audio_data = None
        audio_sr = 24000
        if isinstance(result, dict):
            video_tensor = result.get("x")
            audio_data   = result.get("audio")
            audio_sr     = result.get("audio_sampling_rate", 24000)
        elif isinstance(result, tuple):
            video_tensor = result[0]
            audio_data   = result[1] if len(result) > 1 else None
            audio_sr     = result[2] if len(result) > 2 else 24000
        else:
            video_tensor = result
            audio_data, audio_sr = None, 24000

        if video_tensor is None or not torch.is_tensor(video_tensor):
            yield None, f"❌ No video tensor returned. Got: {type(video_tensor)}"
            return

        if LTX_DIAG:
            # Sampled, so this cannot balloon RAM right after a long generation.
            _bad = _diag_tensor("decoded video", video_tensor)
            if torch.is_tensor(audio_data):
                _diag_tensor("decoded audio", audio_data)
            if _bad:
                print("  [diag] ❌ decoded video contains non-finite values - latent was poisoned.")
                sys.stdout.flush()
            gc.collect()

        video_tensor = video_tensor.cpu()
        gc.collect(); torch.cuda.empty_cache()

        # Save video directly to physical disk OUTPUTS_DIR (not tmpfs RAM)
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        out_filename = f"ltx25_{timestamp}_seed{seed}.mp4"
        out_path = os.path.join(OUTPUTS_DIR, out_filename)

        if video_tensor.dtype != torch.uint8:
            video_tensor = video_tensor.clamp(0, 255).to(torch.uint8)
        if video_tensor.ndim == 4:
            video_tensor = video_tensor.unsqueeze(0)
        save_video(tensor=video_tensor, save_file=out_path, fps=frame_rate, nrow=1)

        # Also copy to /kaggle/working/outputs for external access
        try:
            import shutil
            shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", out_filename))
        except Exception:
            pass

        # ==== Mux native synchronized audio (if generated) ====
        if audio_data is not None:
            try:
                audio_tmp = tempfile.mktemp(suffix=".wav")
                if isinstance(audio_data, np.ndarray):
                    audio_np = audio_data
                    if audio_np.ndim == 2 and audio_np.shape[0] <= 2 and audio_np.shape[1] > audio_np.shape[0]:
                        audio_np = audio_np.T
                    sf.write(audio_tmp, audio_np, int(audio_sr or 24000))
                elif torch.is_tensor(audio_data):
                    import torchaudio
                    cpu_audio = audio_data.cpu().float()
                    if cpu_audio.dim() == 1: cpu_audio = cpu_audio.unsqueeze(0)
                    if cpu_audio.dim() == 3: cpu_audio = cpu_audio.squeeze(0)
                    torchaudio.save(audio_tmp, cpu_audio, int(audio_sr or 24000))

                final_path = out_path.replace(".mp4", "_with_audio.mp4")
                subprocess.run([
                    "ffmpeg", "-y", "-i", out_path, "-i", audio_tmp,
                    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                    "-shortest", final_path
                ], check=True, capture_output=True)
                if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
                    out_path = final_path
                    # Copy audio-muxed version to persistent storage
                    try:
                        shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", os.path.basename(final_path)))
                        shutil.copy2(out_path, os.path.join(OUTPUTS_DIR, os.path.basename(final_path)))
                    except Exception:
                        pass
                    print(f"  ✅ Synchronized audio muxed into output: {out_path}")
            except Exception as e:
                print(f"  ⚠️ Audio mux note: {e}")

        del video_tensor
        gc.collect(); torch.cuda.empty_cache()

        yield out_path, f"✅ Video generated in {elapsed:.1f}s! Seed: {seed} | {width}x{height} | {num_frames} frames | Saved: {out_path}"

    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        sys.stdout.flush()
        gc.collect(); torch.cuda.empty_cache()
        tail = chr(10).join(tb.strip().splitlines()[-4:])
        yield None, f"❌ {type(e).__name__}: {e}" + chr(10) + tail

# ==== GRADIO UI (AIQUEST BRANDED) ====
CSS = '''@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1050px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white !important; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary, #gen-btn { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
'''

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="LTX-2.5 22B Distilled - AIQUEST Academy") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎬 LTX-2.5 22B Distilled - Video Generator</div><div class="brand-subtitle">Created by <strong>AIQUEST Academy</strong> | Kaggle Dual T4 GPU Edition</div><div class="social-buttons"><a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    gr.Markdown(
        '''**Single-Stage 8-Step Distilled Pipeline** | Official INT8 Convrot (~19.5 GB) | Persistent In-Memory Model

⚡ **Performance Presets:** Use **Fast Preview (384p, 2s/3s)** for rapid ~1-2 minute generation. All generated videos are automatically saved to `/kaggle/working/outputs/`.'''
    )

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="🎨 Prompt", lines=3,
                placeholder="A cinematic 4k close-up shot of a majestic snow leopard in the Himalayas during a snowfall, breathing softly, ultra realistic..."
            )

            with gr.Accordion("🖼️ Image to Video (Optional)", open=False):
                with gr.Row():
                    input_image_start = gr.Image(type="filepath", label="🎬 Start Frame (First Frame)", height=180)
                    input_image_end   = gr.Image(type="filepath", label="🎬 End Frame (Last Frame - Optional)", height=180)
                with gr.Row():
                    tail_clip = gr.Video(
                        label="🎞️ Motion Context Clip (short clip from the end of the previous "
                              "scene - carries real motion/velocity, not just a still frame; "
                              "overrides Start Frame when supplied)",
                    )
                with gr.Row():
                    input_ref_images = gr.Image(
                        type="filepath",
                        label="🪪 Reference Image (persistent character/prop identity for this "
                              "scene - independent of Start Frame/Motion Context Clip, both "
                              "can be used together)",
                        height=180,
                    )
                    image_mode = gr.Checkbox(
                        label="🖼️ Reference-Image Generation Mode (automation-only: produces a "
                              "single still at minimum cost instead of a video; not meant to be "
                              "ticked by hand)",
                        value=False,
                    )
                gr.Markdown(
                    '''*• **Start Frame only** → Image-to-Video (model generates from your image)*
*• **Both frames** → First+Last frame interpolation (model generates in-between)*
*• **Motion Context Clip** → continues real motion from a short clip, not a single frame
(takes priority over Start Frame when both are supplied)*
*• **Neither** → Pure Text-to-Video*

*When a frame is supplied the output resolution follows **its** aspect ratio (at the selected
quality preset's pixel budget), so nothing gets cropped — the Aspect Ratio box is then ignored.*'''
                )

            with gr.Row():
                duration_dropdown = gr.Dropdown(
                    label="⏱️ Duration",
                    choices=[
                        "2 Seconds (49 frames - Fast)",
                        "3 Seconds (73 frames - Standard)",
                        "5 Seconds (121 frames - Long)",
                        "8 Seconds (193 frames)",
                        "10 Seconds (241 frames)",
                    ],
                    value="5 Seconds (121 frames - Long)",
                )
                resolution_dropdown = gr.Dropdown(
                    label="📐 Resolution Preset",
                    choices=[
                        "Fast Preview (384p - ~1-2 min)",
                        "Balanced (480p - ~3-5 min)",
                        "High Quality (704p - ~6-8 min)",
                        "Cinema 1080p (1088p - High Detail)",
                    ],
                    value="Balanced (480p - ~3-5 min)",
                )
                aspect_ratio_dropdown = gr.Dropdown(
                    label="📏 Aspect Ratio",
                    choices=["16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                    value="16:9 Landscape",
                )

            # Video CFG is inert on the distilled pipeline: it uses simple_denoising_func
            # and, with NAG_scale <= 1, never encodes a negative prompt - so there is no
            # unconditional branch for CFG to act on. Pinned at 1.0 instead of shown.
            guide_scale = gr.State(1.0)

            # Chained-scene consistency control: lower values keep more of the
            # previous scene's final frame (less drift/"melting" on continuation);
            # 1.0 = full creative freedom (identical to every scene before this
            # control existed). Has no effect on scenes with no start frame.
            with gr.Row():
                input_video_strength = gr.Slider(
                    label="🔗 Start-Frame Influence (lower = stays closer to the previous scene)",
                    minimum=0.0, maximum=1.0, step=0.05, value=1.0,
                )

            with gr.Row():
                audio_cfg = gr.Slider(
                    label="🔊 Audio CFG Scale",
                    minimum=1.0, maximum=5.0, step=0.5, value=1.0,
                )

            with gr.Row():
                num_steps = gr.Slider(
                    label="⚡ Denoising Steps (8 = full quality, 6 ≈ 25% faster)",
                    minimum=4, maximum=8, step=1, value=8,
                )
                seed = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)

            with gr.Row():
                gen_btn   = gr.Button("🎬 Generate Video", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn  = gr.Button("🛑 Stop",            variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear",           variant="secondary", size="lg", elem_id="clear-btn")

        with gr.Column(scale=1):
            video_out  = gr.Video(label="🎥 Generated Video with Synchronized Audio", height=420)
            status_out = gr.Textbox(label="ℹ️ Live Progress & Output Path", interactive=False, lines=5)

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0; text-align: center;">🎬 Created by <strong>AIQUEST Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af; text-align: center;">Official LTX-2.5 22B Distilled INT8 Convrot | Kaggle Dual GPU (GPU T4 x2)</p><p style="font-size: 13px; margin: 10px 0; text-align: center;"><a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> • <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a></p></div>')

    gen_event = gen_btn.click(
        fn=Video_Generation,
        inputs=[prompt, input_image_start, input_image_end, seed, duration_dropdown,
                resolution_dropdown, aspect_ratio_dropdown, guide_scale, audio_cfg, num_steps,
                input_video_strength, tail_clip, image_mode, input_ref_images],
        outputs=[video_out, status_out],
    )
    stop_btn.click(fn=None, cancels=[gen_event])
    clear_btn.click(
        fn=lambda: (None, None, None, "", -1),
        outputs=[input_image_start, input_image_end, video_out, prompt, seed],
    )

print("\\n🚀 Launching AIQUEST Academy Gradio Web Interface...")
sys.stdout.flush()
demo.queue(max_size=20, default_concurrency_limit=1)

allowed_paths = [
    os.path.abspath("outputs"),
    os.path.abspath(WAN2GP_DIR),
    "/kaggle/working",
    "/kaggle/working/outputs",
    "/kaggle/working/Wan2GP",
    "/tmp",
    tempfile.gettempdir()
]

demo.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True,
    max_threads=4,
    ssr_mode=False,
    allowed_paths=allowed_paths
)
"""

with open("run_aiquest_ltx25.py", "w", encoding="utf-8") as f:
    f.write(aiquest_app_code)

print("✅ run_aiquest_ltx25.py created successfully!")

In [ ]:
#@title Step 3.5: Hotfix - High-Speed FP16 + INT8 ConvRot + Full Audio/Video VAE Fix (Run THIS then Step 4)
import os, re

# ============================================================
# CRITICAL FIX 1: Patch distilled.py to run in FP16 on T4 (24s/step)
# ============================================================
for base_dir in ["Wan2GP", "."]:
    distilled_file = os.path.join(base_dir, "models", "ltx2", "ltx_pipelines", "distilled.py")
    if os.path.isfile(distilled_file):
        with open(distilled_file, "r", encoding="utf-8") as f:
            dist_code = f.read()
        dist_code = dist_code.replace("self.dtype = torch.bfloat16", "self.dtype = getattr(models, 'dtype', torch.float16)")
        dist_code = dist_code.replace("dtype = torch.bfloat16", "dtype = getattr(self, 'dtype', torch.float16)")
        with open(distilled_file, "w", encoding="utf-8") as f:
            f.write(dist_code)
        print(f"✅ Patched distilled.py for high-speed FP16 in {distilled_file}")

# ============================================================
# CRITICAL FIX 2: Patch CausalConv3d in Wan2GP video_vae on disk
# Prevents any BFloat16 vs Half dtype mismatch in VAE 3D conv layers
# ============================================================
for base_dir in ["Wan2GP", "."]:
    conv_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "video_vae", "convolution.py")
    if os.path.isfile(conv_file):
        with open(conv_file, "r", encoding="utf-8") as f:
            conv_code = f.read()
        if "x = self.conv(x)" in conv_code and "x.dtype != self.conv.weight.dtype" not in conv_code:
            conv_code = conv_code.replace(
                "        x = self.conv(x)",
                "        if hasattr(self, 'conv') and hasattr(self.conv, 'weight') and x.dtype != self.conv.weight.dtype:\n            x = x.to(dtype=self.conv.weight.dtype)\n        x = self.conv(x)"
            )
            with open(conv_file, "w", encoding="utf-8") as f:
                f.write(conv_code)
            print(f"✅ Patched VAE CausalConv3d auto-cast in {conv_file}")

# ============================================================
# CRITICAL FIX 3: Patch audio_vae.py & vocoder.py in Wan2GP on disk
# Prevents any buffer/weight dtype mismatch during audio decoding
# ============================================================
for base_dir in ["Wan2GP", "."]:
    audio_vae_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "audio_vae", "audio_vae.py")
    if os.path.isfile(audio_vae_file):
        with open(audio_vae_file, "r", encoding="utf-8") as f:
            audio_vae_code = f.read()
        if "decoded_audio = audio_decoder(latent)" in audio_vae_code and "latent.to(dtype=" not in audio_vae_code:
            audio_vae_code = audio_vae_code.replace(
                "    decoded_audio = audio_decoder(latent)",
                "    if hasattr(audio_decoder, 'conv_in') and hasattr(audio_decoder.conv_in, 'weight'):\n        latent = latent.to(dtype=audio_decoder.conv_in.weight.dtype)\n    decoded_audio = audio_decoder(latent)\n    if hasattr(vocoder, 'vocoder') and hasattr(vocoder.vocoder, 'conv_pre'):\n        decoded_audio = decoded_audio.to(dtype=vocoder.vocoder.conv_pre.weight.dtype)"
            )
            with open(audio_vae_file, "w", encoding="utf-8") as f:
                f.write(audio_vae_code)
            print(f"✅ Patched audio_vae.py auto-cast in {audio_vae_file}")

    vocoder_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "audio_vae", "vocoder.py")
    if os.path.isfile(vocoder_file):
        with open(vocoder_file, "r", encoding="utf-8") as f:
            vocoder_code = f.read()
        vocoder_code = vocoder_code.replace(
            "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)",
            "filt = self.filter.to(dtype=x.dtype, device=x.device).expand(n_channels, -1, -1)\n        return F.conv1d(x, filt, stride=self.stride, groups=n_channels)"
        )
        vocoder_code = vocoder_code.replace(
            "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)",
            "basis = self.forward_basis.to(dtype=y.dtype, device=y.device)\n        spec = F.conv1d(y, basis, stride=self.hop_length, padding=0)"
        )
        with open(vocoder_file, "w", encoding="utf-8") as f:
            f.write(vocoder_code)
        print(f"✅ Patched vocoder.py filter & STFT buffers in {vocoder_file}")

# ============================================================
# CRITICAL FIX 3.5: Patch PixelNorm & feature_extractor in FP32
# Prevents FP16 x**2 overflow (>65504 -> inf -> zero/gray background)
# ============================================================
for base_dir in ["Wan2GP", "."]:
    norm_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "model", "common", "normalization.py")
    if os.path.isfile(norm_file):
        with open(norm_file, "r", encoding="utf-8") as f:
            norm_code = f.read()
        if "x_float = x.float()" not in norm_code:
            norm_code = norm_code.replace(
                "        mean_sq = torch.mean(x**2, dim=self.dim, keepdim=True)\n        # Normalize by the root-mean-square (RMS).\n        rms = torch.sqrt(mean_sq + self.eps)\n        return x / rms",
                "        orig_dtype = x.dtype\n        x_float = x.float()\n        mean_sq = torch.mean(x_float**2, dim=self.dim, keepdim=True)\n        rms = torch.sqrt(mean_sq + self.eps)\n        return (x_float / rms).to(orig_dtype)"
            )
            with open(norm_file, "w", encoding="utf-8") as f:
                f.write(norm_code)
            print(f"✅ Patched PixelNorm float32 calculation (fixes gray background) in {norm_file}")

    fe_file = os.path.join(base_dir, "models", "ltx2", "ltx_core", "text_encoders", "gemma", "feature_extractor.py")
    if os.path.isfile(fe_file):
        with open(fe_file, "r", encoding="utf-8") as f:
            fe_code = f.read()
        if "enc_f = encoded_text.float()" not in fe_code:
            fe_code = fe_code.replace(
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    b, t, d, l = encoded_text.shape\n    variance = torch.mean(encoded_text**2, dim=2, keepdim=True)\n    normed = encoded_text * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))",
                "def _norm_and_concat_per_token_rms(encoded_text: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    orig_dtype = encoded_text.dtype\n    enc_f = encoded_text.float()\n    b, t, d, l = enc_f.shape\n    variance = torch.mean(enc_f**2, dim=2, keepdim=True)\n    normed = enc_f * torch.rsqrt(variance + 1e-6)\n    normed = normed.reshape(b, t, d * l).to(orig_dtype)\n    mask_3d = attention_mask.bool().unsqueeze(-1)\n    return torch.where(mask_3d, normed, torch.zeros_like(normed))"
            )
            with open(fe_file, "w", encoding="utf-8") as f:
                f.write(fe_code)
            print(f"✅ Patched feature_extractor float32 RMS precision in {fe_file}")

patch_file = "run_aiquest_ltx25.py"
with open(patch_file, "r", encoding="utf-8") as f:
    code = f.read()

# Remove any previously injected monkey-patches
if "# === AIQUEST HOTFIX" in code:
    code = re.sub(r'\n# === AIQUEST HOTFIX.*?# === END HOTFIX ===\n', '\n', code, flags=re.DOTALL)
    print("✅ Cleaned previous monkey-patch from script")

# ============================================================
# CRITICAL FIX 4: Register Wan2GP INT8 ConvRot handlers with mmgp
# ============================================================
quant_router_init = '''from mmgp import offload, quant_router
_HANDLER_MODULES = [
    "shared.qtypes.scaled_fp8",
    "shared.qtypes.nvfp4",
    "shared.qtypes.bnb_nf4",
    "shared.qtypes.nunchaku_int4",
    "shared.qtypes.nunchaku_fp4",
    "shared.qtypes.asym_w4a8_int8",
    "shared.qtypes.int8_convrot",
    "shared.qtypes.gguf",
]
quant_router.unregister_handler(".fp8_quanto_bridge")
for handler in _HANDLER_MODULES:
    quant_router.register_handler(handler)
from shared.qtypes import gguf as gguf_handler
quant_router.register_file_extension("gguf", gguf_handler)'''

if "quant_router.register_handler" not in code:
    code = code.replace("from mmgp import offload", quant_router_init)
    print("✅ Injected Wan2GP INT8 ConvRot quant_router registration")
else:
    print("✓ quant_router handlers already registered")

code = code.replace("MODEL_DTYPE = torch.bfloat16", "MODEL_DTYPE = torch.float16")
code = code.replace("VAE_DTYPE   = torch.bfloat16", "VAE_DTYPE   = torch.float16")
code = code.replace("convertWeightsFloatTo=torch.bfloat16", "convertWeightsFloatTo=torch.float16")
code = code.replace("profile_no=5,", "profile_no=4,\n    pinnedMemory=\"transformer\",\n    partialPinning=True,\n    perc_reserved_mem_max=0.35,\n    asyncTransfers=True,")
code = code.replace("pinnedMemory=False,", "pinnedMemory=\"transformer\",\n    partialPinning=True,\n    perc_reserved_mem_max=0.35,")
if "partialPinning" not in code:
    code = code.replace("pinnedMemory=\"transformer\",", "pinnedMemory=\"transformer\",\n    partialPinning=True,\n    perc_reserved_mem_max=0.35,")
if "convertWeightsFloatTo" not in code:
    code = code.replace(
        '    quantizeTransformer=False,',
        '    quantizeTransformer=False,\n    convertWeightsFloatTo=torch.float16,'
    )

# Fix 6: Set proper SDPA backends for T4 (Turing GPU uses mem_efficient_sdp)
if 'torch.backends.cuda.enable_flash_sdp(False)' not in code:
    code = code.replace(
        'torch.backends.cuda.enable_flash_sdp(True)',
        'torch.backends.cuda.enable_flash_sdp(False)\ntorch.backends.cuda.enable_mem_efficient_sdp(True)'
    )

# Fix 7: VAE_tile_size=0 (eliminates 256px tile seam boundary artifacts) & Solver="distilled_8_steps"
code = code.replace('VAE_tile_size=256,', 'VAE_tile_size=0,')
code = code.replace('sample_solver="distilled_8_steps_ancestral",', 'sample_solver="distilled_8_steps",')

# Fix 8: Save output to /tmp/ so Gradio can serve it (fixes InvalidPathError)
code = code.replace(
    '        # Save video to outputs directory\n'
    '        timestamp = time.strftime("%Y%m%d_%H%M%S")\n'
    '        out_filename = f"ltx25_{timestamp}_seed{seed}.mp4"\n'
    '        out_path = os.path.join(OUTPUTS_DIR, out_filename)',
    '        # Save video to /tmp/ for Gradio compatibility, then copy to persistent storage\n'
    '        timestamp = time.strftime("%Y%m%d_%H%M%S")\n'
    '        out_filename = f"ltx25_{timestamp}_seed{seed}.mp4"\n'
    '        TMP_OUTPUTS = os.path.join(tempfile.gettempdir(), "ltx25_outputs")\n'
    '        os.makedirs(TMP_OUTPUTS, exist_ok=True)\n'
    '        out_path = os.path.join(TMP_OUTPUTS, out_filename)'
)

# Fix 9: Copy to both persistent dirs
code = code.replace(
    '        # Also copy to /kaggle/working/outputs for external reference\n'
    '        try:\n'
    '            import shutil\n'
    '            shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", out_filename))\n'
    '        except Exception:\n'
    '            pass',
    '        # Copy to persistent storage directories\n'
    '        try:\n'
    '            import shutil\n'
    '            shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", out_filename))\n'
    '            shutil.copy2(out_path, os.path.join(OUTPUTS_DIR, out_filename))\n'
    '        except Exception:\n'
    '            pass'
)

# Fix 10: Also copy audio-muxed files to Wan2GP outputs
old_audio = '                    try:\n                        shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", os.path.basename(final_path)))\n                    except Exception:\n                        pass'
new_audio = '                    try:\n                        shutil.copy2(out_path, os.path.join("/kaggle/working/outputs", os.path.basename(final_path)))\n                        shutil.copy2(out_path, os.path.join(OUTPUTS_DIR, os.path.basename(final_path)))\n                    except Exception:\n                        pass'
code = code.replace(old_audio, new_audio)

# Fix 11: Direct uint8 video saving (avoids per-frame make_grid normalize blowout)
code = code.replace(
    '        video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0\n'
    '        save_video(tensor=video_for_save, save_file=out_path, fps=frame_rate, normalize=True, value_range=(-1, 1))',
    '        if video_tensor.dtype != torch.uint8:\n'
    '            video_tensor = video_tensor.clamp(0, 255).to(torch.uint8)\n'
    '        if video_tensor.ndim == 4:\n'
    '            video_tensor = video_tensor.unsqueeze(0)\n'
    '        save_video(tensor=video_tensor, save_file=out_path, fps=frame_rate, nrow=1)'
)
code = code.replace('del video_tensor, video_for_save', 'del video_tensor')

# Fix 12: Set Video CFG default to 1.0 (Distilled models require CFG=1.0) & Audio CFG default to 1.0
code = code.replace(
    'label="🎯 Video CFG Scale",\n                    minimum=1.0, maximum=7.0, step=0.5, value=3.0,',
    'label="🎯 Video CFG Scale (Distilled: 1.0 recommended)",\n                    minimum=1.0, maximum=3.0, step=0.1, value=1.0,'
)
code = code.replace(
    'label="🔊 Audio CFG Scale",\n                    minimum=1.0, maximum=10.0, step=0.5, value=7.0,',
    'label="🔊 Audio CFG Scale",\n                    minimum=1.0, maximum=5.0, step=0.5, value=1.0,'
)

# Fix 13: Default duration → 5 Seconds, Resolution → 480p
code = code.replace(
    'value="2 Seconds (49 frames - Fast)"',
    'value="5 Seconds (121 frames - Long)"'
)
code = code.replace(
    'value="3 Seconds (73 frames - Standard)"',
    'value="5 Seconds (121 frames - Long)"'
)
code = code.replace(
    'value="Fast Preview (384p - ~1-2 min)"',
    'value="Balanced (480p - ~3-5 min)"'
)

with open(patch_file, "w", encoding="utf-8") as f:
    f.write(code)

print("\n✅ All patches applied successfully to run_aiquest_ltx25.py!")
print("   → High-Speed FP16 enabled on T4 Tensor Cores (~24s/step)")
print("   → VAE_tile_size=0 (disabled spatial tiling seams)")
print("   → sample_solver=distilled_8_steps (clean deterministic Euler flow-matching)")
print("   → Video VAE & Audio Vocoder buffer auto-casts applied")
print("   → Video saving fixed (direct uint8 frames without make_grid distortion)")
print("   → Video CFG Scale set to 1.0 (eliminates blowout / posterization artifacts)")
print("   → Audio CFG Scale set to 1.0")
print("   → Default: 5 Seconds at 480p")
print("\n🔄 Now re-run Step 4 to launch the server.")

In [ ]:
# === Run this AFTER Step 3 (it needs run_aiquest_ltx25.py to already exist) ===

with open("run_aiquest_ltx25.py", "r", encoding="utf-8") as f:
    code = f.read()

# This must match EXACTLY what's at the bottom of your Step 3 code.
old_launch = '''demo.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True,
    max_threads=4,
    ssr_mode=False,
    allowed_paths=allowed_paths
)'''

new_launch = '''demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=False,
    inline=False,
    debug=True,
    show_error=True,
    max_threads=4,
    ssr_mode=False,
    allowed_paths=allowed_paths,
    prevent_thread_lock=True,
)'''

if old_launch not in code:
    raise ValueError(
        "Could not find the expected demo.launch(...) block — your Step 3 "
        "code may be formatted slightly differently than what I assumed. "
        "Open run_aiquest_ltx25.py, search for 'demo.launch(', and paste "
        "that exact block back to me so I can match it precisely."
    )

headless_code = code.replace(old_launch, new_launch)

with open("run_aiquest_ltx25_headless.py", "w", encoding="utf-8") as f:
    f.write(headless_code)

print("✅ run_aiquest_ltx25_headless.py created — this is the file "
      "kaggle_headless_step4.py runs.")


In [ ]:
#@title Step 4: Headless Batch — Job Ingestion, Chaining, Curated Output, Auto-Shutdown
# Run Steps 1 -> 3.5 first (env, cached weights, app script, hotfixes), then the
# launch-patch cell that produces run_aiquest_ltx25_headless.py, THEN this cell.
#
# What this cell does:
#   1. Pulls the job batch the backend published as a Kaggle dataset (jobs.json).
#   2. Launches the generation server in the background and drives it over its API.
#   3. Chains each scene's start frame from the previous scene's last frame when
#      the job asks for it, extracting a non-black frame with retry/backoff.
#   4. Writes ONLY the generated scene videos (in generation order) + manifest.json
#      to final_output/ - nothing else (no debug frames, no server files).
#   5. Cleans up and lets the kernel finish naturally - Kaggle has no "stop" API,
#      so a notebook only ends by running out of cells with nothing left blocking.
#
# Tunables are all in one place below; the logic beneath them shouldn't need touching.

import os
import sys
import json
import time
import glob
import shutil
import logging
import threading
import subprocess

# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #
JOB_DATASET_SLUG = "ltx-job-request"
FINAL_OUTPUT_DIR = "/kaggle/working/final_output"
DEBUG_DIR = "/kaggle/working/_debug_frames"
LOG_FILE = "/kaggle/working/pipeline.log"

PER_JOB_TIMEOUT_S = 1800           # give up on a single scene after this long
SESSION_TIME_BUDGET_S = 8 * 3600   # stop STARTING new scenes past this cumulative wall-clock
SHOW_SERVER_LOG = True

# How closely a CHAINED scene sticks to the previous scene's last frame.
# 1.0 = full creative freedom (what every scene used before this existed).
# Lower = less drift/"melting" across the cut, but less new motion too - this
# is a genuine visual-quality trade-off, not a free win, so it needs real
# testing against your own footage rather than trusting this number blindly.
# Only applies when a scene is actually chaining AND the previous video was
# usable; untouched for fresh/non-chained scenes regardless of this value.
CHAINED_SCENE_INPUT_VIDEO_STRENGTH = 0.75

# How much of the END of the previous scene to carry forward as real motion
# context (not just a still frame) - this is what lets the next scene continue
# an actual trajectory instead of guessing direction from text alone. Tried
# first on every chained scene; falls back to the old single-frame extraction
# below only if this fails outright (ffmpeg error, clip too short, etc.).
TAIL_CLIP_SECONDS = 0.5

# Reference assets (persistent character/prop images reused across scenes) - kept
# in their own working directory, separate from both final_output/ and the
# tail-clip debug directory, per the architecture review: a reference image is
# never the same thing as "a previous scene's last frame" and must not be
# conflated with it. Unlike the tail-clip debug dir, this one is NOT deleted at
# the end of the run - it gets copied into final_output/ so the backend can keep
# the images (and the prompts that produced them) for future reuse without
# spending compute again on a matching request.
REFERENCE_ASSETS_DIR = "/kaggle/working/_reference_assets"

# --------------------------------------------------------------------------- #
# Logging - console + file, both timestamped. Kaggle only captures stdout, so
# everything also goes there, but the file gives a clean copy of just this
# run's own record without hunting through the whole cell output.
# --------------------------------------------------------------------------- #
logger = logging.getLogger("aiquest_batch")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S")

_console = logging.StreamHandler(sys.stdout)
_console.setFormatter(_fmt)
logger.addHandler(_console)

_file = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
_file.setFormatter(_fmt)
logger.addHandler(_file)


def log(msg, level=logging.INFO):
    logger.log(level, msg)


def gpu_mem_snapshot():
    """System-wide GPU memory, regardless of which process holds it - lets you
    see if usage is growing across the batch even though the model itself
    lives in a different process than this one."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        )
        if result.returncode != 0:
            return "nvidia-smi unavailable"
        parts = []
        for ln in result.stdout.strip().splitlines():
            idx, used, total = [x.strip() for x in ln.split(",")]
            parts.append(f"GPU{idx}: {used}/{total} MiB")
        return " | ".join(parts)
    except Exception as e:
        return f"nvidia-smi check failed: {e}"


# --------------------------------------------------------------------------- #
# Job ingestion - backend's published dataset, unchanged schema:
# {"jobs": [{"job_id", "prompt", "duration_seconds", "chain_from_previous", ...}]}
# --------------------------------------------------------------------------- #
import kaggle
OWNER_USERNAME = kaggle.api.get_config_value("username")

job_dataset_id = f"{OWNER_USERNAME}/{JOB_DATASET_SLUG}"
job_dir = "job_request"
os.makedirs(job_dir, exist_ok=True)

log(f"Downloading job batch from Kaggle dataset {job_dataset_id}...")
subprocess.run(["kaggle", "datasets", "download", "-d", job_dataset_id,
                "-p", job_dir, "--unzip", "--force"], check=True)

with open(os.path.join(job_dir, "jobs.json")) as f:
    jobs_payload = json.load(f)
jobs = jobs_payload["jobs"]

_declared_assets = jobs_payload.get("reference_assets", [])
log(f"Loaded {len(jobs)} job(s) and {len(_declared_assets)} reference asset(s) "
    f"from {job_dataset_id}")
for i, job in enumerate(jobs):
    jid = job.get("job_id", f"job{i}")
    log(f"  [{jid}] chain_from_previous={job.get('chain_from_previous', False)}: "
        f"{job.get('prompt', '')[:60]}...")

os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

# --------------------------------------------------------------------------- #
# Server lifecycle - launch as a background process, drive it over its own
# Gradio API. Factored out so a mid-batch timeout can restart it once.
# --------------------------------------------------------------------------- #
def _launch_server():
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        [sys.executable, "-u", "run_aiquest_ltx25_headless.py"],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

    ready = False
    for line in proc.stdout:
        log("[startup] " + line.rstrip())
        if "Running on local URL" in line:
            ready = True
            break

    if not ready:
        raise RuntimeError("App process ended before the local server came up - "
                            "check the [startup] lines above for the real error.")

    def _drain(p):
        try:
            for server_line in p.stdout:
                if SHOW_SERVER_LOG:
                    log("[server] " + server_line.rstrip())
        except Exception:
            pass

    threading.Thread(target=_drain, args=(proc,), daemon=True).start()
    time.sleep(5)

    from gradio_client import Client
    return proc, Client("http://127.0.0.1:7860")


def _shutdown_server(proc):
    if proc is None:
        return
    try:
        proc.terminate()
        proc.wait(timeout=15)
    except Exception:
        try:
            proc.kill()
        except Exception:
            pass


def restart_server(old_proc):
    log("Restarting generation server...", logging.WARNING)
    _shutdown_server(old_proc)
    return _launch_server()


# --------------------------------------------------------------------------- #
# Duration + chaining helpers
# --------------------------------------------------------------------------- #
DURATION_LABELS = {
    2: "2 Seconds (49 frames - Fast)",
    3: "3 Seconds (73 frames - Standard)",
    5: "5 Seconds (121 frames - Long)",
    8: "8 Seconds (193 frames)",
    10: "10 Seconds (241 frames)",
}

ASPECT_RATIO_LABELS = {
    "16:9": "16:9 Landscape",
    "4:3": "4:3 Standard",
    "1:1": "1:1 Square",
    "3:4": "3:4 Portrait",
    "9:16": "9:16 Portrait",
}


def resolve_aspect_ratio_label(aspect_ratio):
    if not aspect_ratio:
        return "1:1 Square"
    if aspect_ratio in ASPECT_RATIO_LABELS.values():  # already a full label
        return aspect_ratio
    if aspect_ratio in ASPECT_RATIO_LABELS:
        return ASPECT_RATIO_LABELS[aspect_ratio]
    log(f"Unrecognized aspect_ratio={aspect_ratio!r} for a reference asset - "
        f"falling back to 1:1 Square", logging.WARNING)
    return "1:1 Square"


def resolve_duration_label(job):
    if "duration" in job:  # already a label string, pass through
        return job["duration"]
    secs = job.get("duration_seconds")
    if secs in DURATION_LABELS:
        return DURATION_LABELS[secs]
    log(f"Unrecognized duration_seconds={secs!r} - falling back to 2s preset", logging.WARNING)
    return DURATION_LABELS[2]


def get_video_duration(video_path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", video_path],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {video_path}: {result.stderr.strip()}")
    try:
        return float(result.stdout.strip())
    except ValueError:
        raise RuntimeError(f"ffprobe returned no duration for {video_path}: {result.stdout!r}")


def is_frame_mostly_black(image_path, threshold=10):
    """threshold is average grayscale brightness (0-255) below which a
    frame is considered suspiciously black."""
    try:
        from PIL import Image
        import numpy as np
        arr = np.array(Image.open(image_path).convert("L"))
        return arr.mean() < threshold
    except Exception:
        return False  # if the check itself fails, don't block on it


def extract_last_frame(video_path, out_image_path):
    """Seeks to a precise offset before the video's real end (via ffprobe's
    duration, not a heuristic) and validates the result isn't black before
    accepting it - backing off further from the end and retrying if it is."""
    duration = get_video_duration(video_path)
    backoff_offsets = [0.08, 0.3, 0.6, 1.0]
    last_error = None

    for offset in backoff_offsets:
        seek_time = max(duration - offset, 0)
        result = subprocess.run(
            ["ffmpeg", "-y", "-ss", f"{seek_time:.3f}", "-i", video_path,
             "-frames:v", "1", "-q:v", "2", "-pix_fmt", "yuvj420p", out_image_path],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            last_error = result.stderr[-500:]
            continue
        if not os.path.exists(out_image_path) or os.path.getsize(out_image_path) < 1024:
            last_error = "output file missing or suspiciously small"
            continue
        if is_frame_mostly_black(out_image_path):
            log(f"Frame at -{offset}s looks black - trying earlier...", logging.WARNING)
            last_error = f"frame at -{offset}s was black"
            continue
        log(f"Extracted usable frame at -{offset:.2f}s before end (duration={duration:.2f}s)")
        return

    raise RuntimeError(f"Could not extract a usable non-black frame after "
                        f"{len(backoff_offsets)} attempts. Last issue: {last_error}")


def extract_tail_clip(video_path, out_clip_path, tail_seconds=TAIL_CLIP_SECONDS):
    """Extracts the last `tail_seconds` of video_path as a short, audio-free clip.
    Unlike extract_last_frame, this is NOT a reimplementation of the same thing -
    a single still frame carries position/appearance at one instant only; this
    carries actual inter-frame motion (direction, speed, rotation) because the
    model receives several consecutive real frames, not one frame plus a text
    guess about what happens next. Raises on failure so the caller can fall back
    to the single-frame mechanism rather than silently degrading."""
    duration = get_video_duration(video_path)
    seek_time = max(duration - tail_seconds, 0)
    result = subprocess.run(
        ["ffmpeg", "-y", "-ss", f"{seek_time:.3f}", "-i", video_path,
         "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", out_clip_path],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg tail-clip extraction failed: {result.stderr[-500:]}")
    if not os.path.exists(out_clip_path) or os.path.getsize(out_clip_path) < 1024:
        raise RuntimeError("tail-clip output missing or suspiciously small")

    clip_duration = get_video_duration(out_clip_path)
    if clip_duration <= 0:
        raise RuntimeError("tail-clip output has zero duration")

    log(f"Extracted tail clip: {clip_duration:.2f}s from the end of "
        f"{os.path.basename(video_path)} (requested {tail_seconds}s, "
        f"source duration {duration:.2f}s)")
    return clip_duration


# --------------------------------------------------------------------------- #
# Reference assets - persistent character/prop images, prepared once per
# production BEFORE the main scene loop, then resolved per-scene. Deliberately
# its own phase rather than folded into the scene loop: a production with no
# reference_assets declared skips this entirely and the scene loop behaves
# exactly as it did before this feature existed.
# --------------------------------------------------------------------------- #
reference_manifest = {"assets": []}


def write_reference_manifest():
    os.makedirs(REFERENCE_ASSETS_DIR, exist_ok=True)
    with open(os.path.join(REFERENCE_ASSETS_DIR, "reference_manifest.json"), "w") as f:
        json.dump(reference_manifest, f, indent=2)


def prepare_reference_assets(reference_assets, client, job_dir):
    """Generates or validates every declared reference asset once, before any
    scene runs. Returns (asset_paths, failed_required):
      asset_paths     - {asset_id: resolved file path}, only for assets that
                         ended up usable (generated or supplied successfully).
      failed_required - {asset_id, ...} for assets marked required that could
                         not be prepared - scenes depending on these must fail
                         explicitly rather than proceed without them.
    """
    if not reference_assets:
        log("Reference preparation not required for this production "
            "(no reference_assets declared).")
        return {}, set()

    os.makedirs(REFERENCE_ASSETS_DIR, exist_ok=True)
    log(f"Reference preparation enabled for this production: "
        f"{len(reference_assets)} asset(s) declared.")

    asset_paths = {}
    failed_required = set()

    for asset in reference_assets:
        asset_id = asset.get("asset_id")
        name = asset.get("name", asset_id)
        category = asset.get("category", "unspecified")
        required = asset.get("required", True)
        creation = asset.get("creation", {})
        method = creation.get("method", "generate")

        if not asset_id:
            log(f"Skipping a reference asset with no asset_id: {asset!r}", logging.ERROR)
            continue

        log(f"[{asset_id}] \"{name}\" (category={category}, required={required}, "
            f"method={method}) - preparing...")

        entry = {
            "asset_id": asset_id, "name": name, "category": category,
            "required": required, "method": method,
        }
        start = time.time()

        try:
            if method == "generate":
                prompt = creation.get("prompt", "")
                negative_prompt = creation.get("negative_prompt", "")
                if negative_prompt:
                    # Confirmed earlier in this pipeline's own source: negative
                    # prompts have no effect here (NAG is never enabled, n_prompt
                    # is hardcoded empty at the generate() call). Don't pretend
                    # this field does something it doesn't.
                    log(f"[{asset_id}] negative_prompt was supplied but this "
                        f"pipeline cannot act on it (NAG is not wired up) - "
                        f"ignoring it, not silently dropping it unlogged.",
                        logging.WARNING)

                aspect_label = resolve_aspect_ratio_label(creation.get("aspect_ratio"))
                job_handle = client.submit(
                    prompt=prompt,
                    input_image_start=None, input_image_end=None,
                    seed=-1,
                    duration_dropdown="2 Seconds (49 frames - Fast)",  # overridden internally
                    resolution_dropdown="Fast Preview (384p - ~1-2 min)",  # cheapest preset
                    aspect_ratio_dropdown=aspect_label,
                    input_video_strength=1.0,
                    tail_clip=None,
                    image_mode=True,
                    input_ref_images=None,
                    audio_cfg=1.0,
                    num_steps=8,
                    api_name="/Video_Generation",
                )
                result = job_handle.result(timeout=PER_JOB_TIMEOUT_S)
                video_info, _status_text = result
                generated_video_path = video_info["video"] if isinstance(video_info, dict) else video_info

                out_path = os.path.join(REFERENCE_ASSETS_DIR, f"{asset_id}.png")
                extract_last_frame(generated_video_path, out_path)  # reuses the existing,
                # already-tested single-frame extractor - a 1-frame source clip just
                # means the black-frame retry loop succeeds on its first attempt.

                elapsed = time.time() - start
                entry.update({
                    # Relative to this manifest's own folder (reference_assets/),
                    # not absolute - this manifest gets copied verbatim into
                    # final_output/, and /kaggle/working/ won't exist once the
                    # backend reads it. asset_paths (below) keeps the real
                    # absolute path for this run's own use.
                    "status": "generated", "path": os.path.basename(out_path),
                    "prompt_used": prompt, "aspect_ratio": aspect_label,
                    "elapsed_seconds": round(elapsed, 1),
                })
                asset_paths[asset_id] = out_path
                log(f"[{asset_id}] generated successfully in {elapsed:.1f}s -> {out_path}")

            elif method == "upload":
                uploaded_ref = creation.get("uploaded_image_ref")
                src_path = os.path.join(job_dir, uploaded_ref) if uploaded_ref else None
                if not uploaded_ref or not os.path.isfile(src_path):
                    raise FileNotFoundError(
                        f"uploaded_image_ref={uploaded_ref!r} not found in the job dataset")
                out_path = os.path.join(REFERENCE_ASSETS_DIR, f"{asset_id}{os.path.splitext(uploaded_ref)[1]}")
                shutil.copy2(src_path, out_path)
                entry.update({"status": "supplied", "path": os.path.basename(out_path),
                              "source_file": uploaded_ref})
                asset_paths[asset_id] = out_path
                log(f"[{asset_id}] supplied via upload -> {out_path}")

            else:
                raise ValueError(f"Unrecognized creation.method={method!r} "
                                  f"(expected 'generate' or 'upload')")

        except Exception as e:
            elapsed = time.time() - start
            entry.update({"status": "failed", "error": str(e), "elapsed_seconds": round(elapsed, 1)})
            log(f"[{asset_id}] FAILED ({e})", logging.ERROR)
            if required:
                failed_required.add(asset_id)
                log(f"[{asset_id}] is required=True - every scene referencing it will be "
                    f"marked failed rather than silently proceeding without it.", logging.ERROR)
            else:
                log(f"[{asset_id}] is required=False - scenes referencing it will proceed "
                    f"without this reference.", logging.WARNING)

        reference_manifest["assets"].append(entry)
        write_reference_manifest()  # after every asset, not just at the end - same
        # crash-safety reasoning as the scene manifest.

    log(f"Reference preparation complete: {len(asset_paths)} usable, "
        f"{len(failed_required)} required asset(s) failed.")
    return asset_paths, failed_required


def resolve_scene_reference(job, asset_paths, failed_required):
    """Per-scene reference lookup - mirrors how continuity_mode is resolved per
    scene. Returns (resolved_path_or_None, status_string). Raises RuntimeError
    only when the scene cannot proceed at all (its reference was required and
    failed) - every other outcome is a normal, loggable degradation."""
    requested = job.get("reference_asset_ids") or []
    if not requested:
        return None, "none-requested"

    primary = requested[0]
    if len(requested) > 1:
        log(f"Scene requested {len(requested)} reference assets {requested} - this "
            f"checkpoint supports one reference per scene (one_image_ref_only). "
            f"Using '{primary}', dropping {requested[1:]}.", logging.WARNING)

    if primary in failed_required:
        raise RuntimeError(f"required reference asset '{primary}' failed during preparation")

    if primary in asset_paths:
        return asset_paths[primary], f"resolved:{primary}"

    log(f"Scene referenced asset_id='{primary}' which was never declared in "
        f"reference_assets, or was declared but not required and failed - "
        f"proceeding without a reference.", logging.WARNING)
    return None, f"unresolved:{primary}"


# --------------------------------------------------------------------------- #
# Manifest - written after every job (not just at the end), so a hard crash
# still leaves an accurate record of what actually completed.
# --------------------------------------------------------------------------- #
batch_manifest = {"jobs": [], "reference_assets": []}


def write_manifest():
    with open(os.path.join(FINAL_OUTPUT_DIR, "manifest.json"), "w") as f:
        json.dump(batch_manifest, f, indent=2)


def mark_remaining_skipped(jobs, from_index, reason):
    for idx in range(from_index, len(jobs)):
        skipped = jobs[idx]
        batch_manifest["jobs"].append({
            "sequence": idx,
            "job_id": skipped.get("job_id", f"job{idx}"),
            "status": "skipped",
            "error": reason,
        })


# --------------------------------------------------------------------------- #
# Main batch loop
# --------------------------------------------------------------------------- #
app_process = None
client = None
previous_video_path = None
restarted_once = False
batch_start_time = time.time()

try:
    app_process, client = _launch_server()
    log(f"Server up. Baseline GPU memory: {gpu_mem_snapshot()}")

    try:
        reference_asset_paths, reference_failed_required = prepare_reference_assets(
            jobs_payload.get("reference_assets", []), client, job_dir
        )
    except Exception as ref_prep_fatal:
        # Reference prep itself blew up (not just one asset) - rather than guess
        # which assets would have been fine, treat every declared asset as
        # unavailable. Scenes that don't reference any asset are completely
        # unaffected by this; scenes that do will fail explicitly below instead
        # of silently proceeding without a reference they were told to use.
        log(f"Reference preparation failed entirely: {ref_prep_fatal}", logging.ERROR)
        reference_asset_paths = {}
        reference_failed_required = {
            a.get("asset_id") for a in jobs_payload.get("reference_assets", [])
            if a.get("required", True) and a.get("asset_id")
        }

    batch_manifest["reference_assets"] = reference_manifest["assets"]
    write_manifest()

    i = 0
    while i < len(jobs):
        job = jobs[i]
        job_id = job.get("job_id", f"job{i}")
        seq_name = f"scene_{i:03d}"

        elapsed_session = time.time() - batch_start_time
        if elapsed_session > SESSION_TIME_BUDGET_S:
            log(f"Session time budget ({SESSION_TIME_BUDGET_S}s) reached before "
                f"starting {job_id} - stopping here so this run finishes cleanly. "
                f"{len(jobs) - i} scene(s) remain for a follow-up run.", logging.WARNING)
            mark_remaining_skipped(jobs, i, "session time budget reached")
            write_manifest()
            break

        log(f"=== Generating {job_id} ({i + 1}/{len(jobs)}) - "
            f"elapsed {elapsed_session / 60:.1f} min ===")
        job_handle = None
        job_start = time.time()
        advance = True  # False only when we recover-and-retry via server restart

        try:
            # Continuity resolution, in priority order:
            #   1. tail-clip (real motion context - several consecutive frames)
            #   2. single-frame fallback (today's mechanism - position only, no velocity)
            #   3. none (fresh scene, or chaining requested but nothing usable)
            # continuity_mode is logged and recorded in the manifest every time, so it's
            # always possible to tell - per scene - which of these three actually fired.
            tail_clip_arg = None
            start_frame_arg = None
            continuity_mode = "none"
            reference_path = None
            reference_status = "not-attempted"  # overwritten below unless
            # resolve_scene_reference itself raises, in which case the except
            # blocks still have a defined value to record rather than a NameError.

            if job.get("chain_from_previous") and previous_video_path:
                clip_path = os.path.join(DEBUG_DIR, f"{seq_name}_tail_clip.mp4")
                log(f"Attempting tail-clip extraction ({TAIL_CLIP_SECONDS}s) -> {clip_path}")
                try:
                    extract_tail_clip(previous_video_path, clip_path)
                    from gradio_client import handle_file
                    # gr.Video's API payload is {"video": <FileData>}, not a bare
                    # FileData like gr.Image takes - confirmed directly against
                    # gradio 5.29.0's VideoData model (video: FileData, required).
                    tail_clip_arg = {"video": handle_file(clip_path)}
                    continuity_mode = "tail-clip"
                    log(f"[{job_id}] continuity_mode=tail-clip - real motion context "
                        f"will be used for this scene.")
                except Exception as e:
                    log(f"Tail-clip extraction failed ({e}) - "
                        f"falling back to single-frame extraction.", logging.WARNING)
                    frame_path = os.path.join(DEBUG_DIR, f"{seq_name}_start_frame.jpg")
                    try:
                        extract_last_frame(previous_video_path, frame_path)
                        from gradio_client import handle_file
                        start_frame_arg = handle_file(frame_path)
                        continuity_mode = "single-frame-fallback"
                        log(f"[{job_id}] continuity_mode=single-frame-fallback - "
                            f"position-only continuity, no motion context.", logging.WARNING)
                    except Exception as e2:
                        log(f"Single-frame fallback also failed ({e2}) - "
                            f"generating WITHOUT any continuity input.", logging.WARNING)
                        continuity_mode = "none-extraction-failed"
            elif job.get("chain_from_previous") and not previous_video_path:
                log("chain_from_previous=True but no usable previous scene - "
                    "generating without any continuity input.", logging.WARNING)
                continuity_mode = "none-no-previous-scene"

            # Reference-asset resolution - independent of the continuity resolution
            # above (confirmed in source: separate conditioning channels, no shared
            # state), so both can be active on the same scene. resolve_scene_reference
            # raises if this scene's reference was required and failed to prepare -
            # that propagates to the same except-Exception handling as any other
            # scene failure below, rather than needing its own parallel path.
            reference_path, reference_status = resolve_scene_reference(
                job, reference_asset_paths, reference_failed_required
            )
            if reference_path:
                log(f"[{job_id}] reference_status={reference_status} - "
                    f"using {reference_path}")

            # Only dial down strength when something is actually feeding continuity
            # in (tail-clip or single-frame) - a job that asked to chain but got
            # neither gets full 1.0 freedom, same as always, since there's nothing
            # to "stay close to" anymore.
            has_continuity_input = tail_clip_arg is not None or start_frame_arg is not None
            if has_continuity_input:
                video_strength = job.get("input_video_strength", CHAINED_SCENE_INPUT_VIDEO_STRENGTH)
            else:
                video_strength = job.get("input_video_strength", 1.0)

            log(f"Submitting {job_id} (continuity_mode={continuity_mode}, "
                f"reference_status={reference_status}, "
                f"input_video_strength={video_strength}, timeout {PER_JOB_TIMEOUT_S}s)...")
            # NOTE: no guide_scale kwarg here. This Step 3 pins video CFG to a
            # fixed gr.State(1.0) rather than a real component - the distilled
            # pipeline never uses it, so it isn't part of the app's public API
            # at all. Passing it raises "not a valid keyword argument".
            job_handle = client.submit(
                prompt=job["prompt"],
                input_image_start=start_frame_arg,
                input_image_end=None,
                seed=job.get("seed", -1),
                duration_dropdown=resolve_duration_label(job),
                resolution_dropdown=job.get("resolution", "Fast Preview (384p - ~1-2 min)"),
                aspect_ratio_dropdown=job.get("aspect_ratio", "16:9 Landscape"),
                input_video_strength=video_strength,
                tail_clip=tail_clip_arg,
                input_ref_images=reference_path,
                audio_cfg=job.get("audio_cfg", 1.0),
                num_steps=job.get("num_steps", 8),
                api_name="/Video_Generation",
            )
            result = job_handle.result(timeout=PER_JOB_TIMEOUT_S)
            video_info, status_text = result
            generated_path = video_info["video"] if isinstance(video_info, dict) else video_info
            previous_video_path = generated_path

            has_audio = "_with_audio" in os.path.basename(generated_path)
            dest_name = f"{seq_name}_with_audio.mp4" if has_audio else f"{seq_name}.mp4"
            dest_path = os.path.join(FINAL_OUTPUT_DIR, dest_name)
            shutil.copy2(generated_path, dest_path)

            elapsed_job = time.time() - job_start
            log(f"Saved {dest_name} (has_audio={has_audio}, took {elapsed_job:.0f}s). "
                f"{gpu_mem_snapshot()}")

            batch_manifest["jobs"].append({
                "sequence": i, "job_id": job_id, "file": dest_name,
                "has_audio": has_audio, "status": "succeeded",
                "elapsed_seconds": round(elapsed_job, 1),
                "input_video_strength": video_strength,
                "continuity_mode": continuity_mode,
                "reference_status": reference_status,
            })

        except TimeoutError:
            log(f"{job_id} exceeded {PER_JOB_TIMEOUT_S}s.", logging.ERROR)
            try:
                if job_handle is not None:
                    job_handle.cancel()
            except Exception:
                pass
            batch_manifest["jobs"].append({
                "sequence": i, "job_id": job_id, "status": "failed",
                "error": f"timed out after {PER_JOB_TIMEOUT_S}s",
                "continuity_mode": continuity_mode,
                "reference_status": reference_status,
            })
            previous_video_path = None

            if not restarted_once:
                restarted_once = True
                try:
                    app_process, client = restart_server(app_process)
                    log("Server restarted - continuing with the next scene.")
                except Exception as restart_err:
                    log(f"Server restart failed too ({restart_err}) - "
                        f"aborting remaining batch.", logging.ERROR)
                    write_manifest()
                    mark_remaining_skipped(jobs, i + 1,
                        "earlier scene timed out and the server could not recover")
                    write_manifest()
                    break
            else:
                log("Already used this run's one restart attempt - "
                    "aborting remaining batch.", logging.ERROR)
                write_manifest()
                mark_remaining_skipped(jobs, i + 1,
                    "earlier scene timed out twice; not retrying further")
                write_manifest()
                break

        except Exception as e:
            log(f"{job_id} FAILED: {e}", logging.ERROR)
            batch_manifest["jobs"].append({
                "sequence": i, "job_id": job_id, "status": "failed", "error": str(e),
                "continuity_mode": continuity_mode,
                "reference_status": reference_status,
            })
            previous_video_path = None

        write_manifest()
        i += 1

    succeeded = sum(1 for j in batch_manifest["jobs"] if j["status"] == "succeeded")
    ref_generated = sum(1 for a in reference_manifest["assets"] if a["status"] in ("generated", "supplied"))
    ref_failed = sum(1 for a in reference_manifest["assets"] if a["status"] == "failed")
    total_elapsed = time.time() - batch_start_time
    log(f"Batch complete: {succeeded}/{len(jobs)} job(s) succeeded, "
        f"{ref_generated} reference asset(s) ready / {ref_failed} failed, "
        f"in {total_elapsed / 60:.1f} min.")

except Exception as fatal:
    log(f"FATAL: unhandled error in batch pipeline: {fatal}", logging.ERROR)

finally:
    write_manifest()
    log("Shutting down local server.")
    _shutdown_server(app_process)

    # Reference images (and the manifest carrying their prompts/metadata) ship
    # inside final_output/ - unlike tail-clip debug frames, these are meant for
    # the backend to keep and reuse, not just internal scratch. Copied here, in
    # the finally block, so even a run that errors out part-way through still
    # ships whatever reference assets were actually completed.
    if os.path.isdir(REFERENCE_ASSETS_DIR):
        ref_dest = os.path.join(FINAL_OUTPUT_DIR, "reference_assets")
        shutil.copytree(REFERENCE_ASSETS_DIR, ref_dest, dirs_exist_ok=True)
        log(f"Copied reference assets into {ref_dest} for backend reuse.")

    # Clean up everything except final_output/ (already correctly populated
    # per-job during the loop above) - including the chaining debug frames,
    # which are deliberately kept OUT of final_output/ the whole time.
    shutil.rmtree("/kaggle/working/Wan2GP", ignore_errors=True)
    shutil.rmtree(DEBUG_DIR, ignore_errors=True)
    shutil.rmtree(REFERENCE_ASSETS_DIR, ignore_errors=True)
    if os.path.isdir("/kaggle/working/outputs"):
        shutil.rmtree("/kaggle/working/outputs", ignore_errors=True)
    for pattern in ("job_request", "run_aiquest_ltx25.py", "run_aiquest_ltx25_headless.py", "*.log"):
        for path in glob.glob(f"/kaggle/working/{pattern}"):
            if os.path.isdir(path):
                shutil.rmtree(path, ignore_errors=True)
            elif path != LOG_FILE:
                os.remove(path)

    log("Cleanup complete - /kaggle/working/final_output/ now holds the "
        "generated scene videos (in order), manifest.json, and reference_assets/ "
        "(images + reference_manifest.json) if any were declared for this production.")
    log("Done - this kernel run should now finish naturally.")
